# Baseline Comparison: PureRAG vs TitanOnly vs HybridRAG

This notebook compares three retrieval strategies:
- **PureRAG**: BM25 + Embedding (no Memory)
- **TitanOnly**: Memory-guided retrieval only
- **HybridRAG**: BM25 + Memory + Embedding fusion

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch sentence-transformers transformers tqdm

## 2. Setup File System

In [ ]:
import os
import base64

print(f'Current working directory: {os.getcwd()}')

def write_file(path, b64_content):
    target_path = os.path.abspath(path)
    directory = os.path.dirname(target_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(target_path, 'wb') as f:
        f.write(base64.b64decode(b64_content))
    print(f'Created {target_path}')

files = {
    'src/main.py': '...',
    'data/sample_essays.py': 'IiIiClNhbXBsZSBFc3NheXMgZm9yIFRpdGFuUkFHIEV4cGVyaW1lbnRzCgpUaGVzZSBlc3NheXMgY29udGFpbiBmYWN0dWFsIGluZm9ybWF0aW9uIHRoYXQgY2FuIGJlIHVzZWQgdG8gdGVzdAp0aGUgbW9kZWwncyBhYmlsaXR5IHRvIG1lbW9yaXplIGFuZCByZWNhbGwgc3BlY2lmaWMgZGV0YWlscy4KIiIiCgpFU1NBWV9DTElNQVRFID0gIiIiCiMgVGhlIFNjaWVuY2Ugb2YgQ2xpbWF0ZSBDaGFuZ2UKCiMjIEludHJvZHVjdGlvbgpDbGltYXRlIGNoYW5nZSByZWZlcnMgdG8gbG9uZy10ZXJtIHNoaWZ0cyBpbiBnbG9iYWwgdGVtcGVyYXR1cmVzIGFuZCB3ZWF0aGVyIHBhdHRlcm5zLiAKV2hpbGUgbmF0dXJhbCBmYWN0b3JzIGxpa2Ugdm9sY2FuaWMgZXJ1cHRpb25zIGFuZCBzb2xhciBjeWNsZXMgY2FuIGluZmx1ZW5jZSBjbGltYXRlLCAKaHVtYW4gYWN0aXZpdGllcyBoYXZlIGJlZW4gdGhlIHByaW1hcnkgZHJpdmVyIHNpbmNlIHRoZSAxODAwcy4KCiMjIEtleSBGYWN0cwotIFRoZSBnbG9iYWwgYXZlcmFnZSB0ZW1wZXJhdHVyZSBoYXMgcmlzZW4gYnkgYXBwcm94aW1hdGVseSAxLjHCsEMgc2luY2UgcHJlLWluZHVzdHJpYWwgdGltZXMuCi0gVGhlIGNvbmNlbnRyYXRpb24gb2YgQ08yIGluIHRoZSBhdG1vc3BoZXJlIHJlYWNoZWQgNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMuCi0gU2VhIGxldmVscyBoYXZlIHJpc2VuIGJ5IGFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykgc2luY2UgMTkwMC4KLSBUaGUgQXJjdGljIGlzIHdhcm1pbmcgbmVhcmx5IGZvdXIgdGltZXMgZmFzdGVyIHRoYW4gdGhlIGdsb2JhbCBhdmVyYWdlLgoKIyMgQ2F1c2VzClRoZSBidXJuaW5nIG9mIGZvc3NpbCBmdWVsc+KAlGNvYWwsIG9pbCwgYW5kIG5hdHVyYWwgZ2Fz4oCUcmVsZWFzZXMgY2FyYm9uIGRpb3hpZGUgKENPMikgCmFuZCBvdGhlciBncmVlbmhvdXNlIGdhc2VzIGludG8gdGhlIGF0bW9zcGhlcmUuIFRoZXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIApjcmVhdGluZyBhICJncmVlbmhvdXNlIGVmZmVjdCIgdGhhdCB3YXJtcyB0aGUgcGxhbmV0LgoKIyMgSW1wYWN0cwoxLiAqKkV4dHJlbWUgV2VhdGhlcioqOiBNb3JlIGZyZXF1ZW50IGhlYXR3YXZlcywgZHJvdWdodHMsIGFuZCBpbnRlbnNlIHN0b3Jtcy4KMi4gKipFY29zeXN0ZW0gRGlzcnVwdGlvbioqOiBDb3JhbCBibGVhY2hpbmcsIHNwZWNpZXMgbWlncmF0aW9uLCBhbmQgYmlvZGl2ZXJzaXR5IGxvc3MuCjMuICoqSHVtYW4gSGVhbHRoKio6IEluY3JlYXNlZCBoZWF0LXJlbGF0ZWQgaWxsbmVzc2VzIGFuZCBzcHJlYWQgb2YgdmVjdG9yLWJvcm5lIGRpc2Vhc2VzLgo0LiAqKkFncmljdWx0dXJlKio6IFNoaWZ0cyBpbiBncm93aW5nIHNlYXNvbnMgYW5kIHJlZHVjZWQgY3JvcCB5aWVsZHMgaW4gc29tZSByZWdpb25zLgoKIyMgU29sdXRpb25zCi0gVHJhbnNpdGlvbiB0byByZW5ld2FibGUgZW5lcmd5IHNvdXJjZXMgKHNvbGFyLCB3aW5kLCBoeWRybykuCi0gSW1wcm92ZSBlbmVyZ3kgZWZmaWNpZW5jeSBpbiBidWlsZGluZ3MgYW5kIHRyYW5zcG9ydGF0aW9uLgotIFByb3RlY3QgYW5kIHJlc3RvcmUgZm9yZXN0cywgd2hpY2ggYWJzb3JiIENPMi4KLSBJbXBsZW1lbnQgY2FyYm9uIHByaWNpbmcgdG8gaW5jZW50aXZpemUgZW1pc3Npb24gcmVkdWN0aW9ucy4KCiMjIENvbmNsdXNpb24KQWRkcmVzc2luZyBjbGltYXRlIGNoYW5nZSByZXF1aXJlcyB1cmdlbnQsIGNvb3JkaW5hdGVkIGdsb2JhbCBhY3Rpb24uIFRoZSBQYXJpcyBBZ3JlZW1lbnQgCm9mIDIwMTUgc2V0IHRoZSBnb2FsIG9mIGxpbWl0aW5nIHdhcm1pbmcgdG8gMS41wrBDIGFib3ZlIHByZS1pbmR1c3RyaWFsIGxldmVscy4KIiIiCgpFU1NBWV9BSSA9ICIiIgojIFRoZSBFdm9sdXRpb24gb2YgQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UKCiMjIEhpc3RvcmljYWwgQmFja2dyb3VuZApUaGUgdGVybSAiQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UiIHdhcyBjb2luZWQgYnkgSm9obiBNY0NhcnRoeSBpbiAxOTU2IGF0IHRoZSBEYXJ0bW91dGggCkNvbmZlcmVuY2UuIEVhcmx5IEFJIHJlc2VhcmNoIGZvY3VzZWQgb24gc3ltYm9saWMgcmVhc29uaW5nIGFuZCBwcm9ibGVtLXNvbHZpbmcuCgojIyBUaW1lbGluZSBvZiBNYWpvciBNaWxlc3RvbmVzCi0gKioxOTUwKio6IEFsYW4gVHVyaW5nIHByb3Bvc2VzIHRoZSAiVHVyaW5nIFRlc3QiIGZvciBtYWNoaW5lIGludGVsbGlnZW5jZS4KLSAqKjE5NTYqKjogRGFydG1vdXRoIENvbmZlcmVuY2UgbWFya3MgdGhlIGJpcnRoIG9mIEFJIGFzIGEgZmllbGQuCi0gKioxOTk3Kio6IElCTSdzIERlZXAgQmx1ZSBkZWZlYXRzIHdvcmxkIGNoZXNzIGNoYW1waW9uIEdhcnJ5IEthc3Bhcm92LgotICoqMjAxMSoqOiBJQk0gV2F0c29uIHdpbnMgSmVvcGFyZHkhIGFnYWluc3QgaHVtYW4gY2hhbXBpb25zLgotICoqMjAxNioqOiBHb29nbGUncyBBbHBoYUdvIGRlZmVhdHMgTGVlIFNlZG9sIGluIHRoZSBnYW1lIG9mIEdvLgotICoqMjAyMioqOiBDaGF0R1BUIGxhdW5jaGVzLCBkZW1vbnN0cmF0aW5nIGFkdmFuY2VkIGNvbnZlcnNhdGlvbmFsIEFJLgoKIyMgVHlwZXMgb2YgQUkKMS4gKipOYXJyb3cgQUkgKEFOSSkqKjogRGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIChlLmcuLCBpbWFnZSByZWNvZ25pdGlvbiwgbGFuZ3VhZ2UgdHJhbnNsYXRpb24pLgoyLiAqKkdlbmVyYWwgQUkgKEFHSSkqKjogSHlwb3RoZXRpY2FsIEFJIHdpdGggaHVtYW4tbGV2ZWwgY29nbml0aXZlIGFiaWxpdGllcyBhY3Jvc3MgYWxsIGRvbWFpbnMuCjMuICoqU3VwZXJpbnRlbGxpZ2VudCBBSSAoQVNJKSoqOiBUaGVvcmV0aWNhbCBBSSBzdXJwYXNzaW5nIGh1bWFuIGludGVsbGlnZW5jZSBlbnRpcmVseS4KCiMjIEN1cnJlbnQgQXBwbGljYXRpb25zCi0gKipIZWFsdGhjYXJlKio6IE1lZGljYWwgaW1hZ2luZyBhbmFseXNpcywgZHJ1ZyBkaXNjb3ZlcnksIHBlcnNvbmFsaXplZCB0cmVhdG1lbnQgcGxhbnMuCi0gKipGaW5hbmNlKio6IEZyYXVkIGRldGVjdGlvbiwgYWxnb3JpdGhtaWMgdHJhZGluZywgY3JlZGl0IHNjb3JpbmcuCi0gKipUcmFuc3BvcnRhdGlvbioqOiBBdXRvbm9tb3VzIHZlaGljbGVzLCB0cmFmZmljIG9wdGltaXphdGlvbiwgcm91dGUgcGxhbm5pbmcuCi0gKipFZHVjYXRpb24qKjogUGVyc29uYWxpemVkIGxlYXJuaW5nLCBhdXRvbWF0ZWQgZ3JhZGluZywgdHV0b3Jpbmcgc3lzdGVtcy4KCiMjIEtleSBUZWNobm9sb2dpZXMKVGhlIG1vZGVybiBBSSByZXZvbHV0aW9uIGlzIGJ1aWx0IG9uOgotICoqRGVlcCBMZWFybmluZyoqOiBOZXVyYWwgbmV0d29ya3Mgd2l0aCBtdWx0aXBsZSBsYXllcnMgZm9yIGZlYXR1cmUgZXh0cmFjdGlvbi4KLSAqKlRyYW5zZm9ybWVycyoqOiBBcmNoaXRlY3R1cmUgYmVoaW5kIGxhcmdlIGxhbmd1YWdlIG1vZGVscyBsaWtlIEdQVC00IGFuZCBDbGF1ZGUuCi0gKipSZWluZm9yY2VtZW50IExlYXJuaW5nKio6IFRyYWluaW5nIGFnZW50cyB0aHJvdWdoIHJld2FyZC1iYXNlZCBmZWVkYmFjay4KCiMjIEV0aGljYWwgQ29uc2lkZXJhdGlvbnMKLSBCaWFzIGluIHRyYWluaW5nIGRhdGEgbGVhZGluZyB0byB1bmZhaXIgb3V0Y29tZXMuCi0gSm9iIGRpc3BsYWNlbWVudCBhbmQgZWNvbm9taWMgaW5lcXVhbGl0eS4KLSBQcml2YWN5IGNvbmNlcm5zIGZyb20gZGF0YSBjb2xsZWN0aW9uLgotIFBvdGVudGlhbCBtaXN1c2UgZm9yIG1pc2luZm9ybWF0aW9uIGFuZCBzdXJ2ZWlsbGFuY2UuCgojIyBUaGUgRnV0dXJlCkV4cGVydHMgcHJlZGljdCBBSSB3aWxsIGNvbnRpbnVlIHRvIHRyYW5zZm9ybSBpbmR1c3RyaWVzLCB3aXRoIGVzdGltYXRlZCBlY29ub21pYyBpbXBhY3QgCm9mICQxNS43IHRyaWxsaW9uIGJ5IDIwMzAgYWNjb3JkaW5nIHRvIFB3Qy4gVGhlIGRldmVsb3BtZW50IG9mIEFHSSByZW1haW5zIGEgbG9uZy10ZXJtIGdvYWwuCiIiIgoKRVNTQVlfU1BBQ0UgPSAiIiIKIyBFeHBsb3JpbmcgdGhlIFNvbGFyIFN5c3RlbQoKIyMgT3VyIENvc21pYyBOZWlnaGJvcmhvb2QKVGhlIFNvbGFyIFN5c3RlbSBjb25zaXN0cyBvZiB0aGUgU3VuLCBlaWdodCBwbGFuZXRzLCBkd2FyZiBwbGFuZXRzLCBtb29ucywgYXN0ZXJvaWRzLCAKYW5kIGNvbWV0cy4gSXQgZm9ybWVkIGFwcHJveGltYXRlbHkgNC42IGJpbGxpb24geWVhcnMgYWdvIGZyb20gYSBnaWFudCBtb2xlY3VsYXIgY2xvdWQuCgojIyBUaGUgU3VuCi0gKipUeXBlKio6IEctdHlwZSBtYWluLXNlcXVlbmNlIHN0YXIgKHllbGxvdyBkd2FyZikKLSAqKkRpYW1ldGVyKio6IDEsMzkyLDcwMCBrbSAoMTA5IHRpbWVzIEVhcnRoJ3MgZGlhbWV0ZXIpCi0gKipTdXJmYWNlIFRlbXBlcmF0dXJlKio6IDUsNTAwwrBDICg5LDkwMMKwRikKLSAqKkNvcmUgVGVtcGVyYXR1cmUqKjogMTUgbWlsbGlvbsKwQwotICoqRGlzdGFuY2UgZnJvbSBFYXJ0aCoqOiAxNDkuNiBtaWxsaW9uIGttICgxIEFzdHJvbm9taWNhbCBVbml0KQoKIyMgVGhlIFBsYW5ldHMKIyMjIElubmVyIFBsYW5ldHMgKFJvY2t5L1RlcnJlc3RyaWFsKQoxLiAqKk1lcmN1cnkqKjogU21hbGxlc3QgcGxhbmV0LCA4OC1kYXkgb3JiaXQsIG5vIG1vb25zLgoyLiAqKlZlbnVzKio6IEhvdHRlc3QgcGxhbmV0ICg0NzXCsEMgc3VyZmFjZSksIHJvdGF0ZXMgYmFja3dhcmRzLgozLiAqKkVhcnRoKio6IE9ubHkga25vd24gcGxhbmV0IHdpdGggbGlxdWlkIHdhdGVyIG9uIHN1cmZhY2UgYW5kIGxpZmUuCjQuICoqTWFycyoqOiAiUmVkIFBsYW5ldCIsIGhhcyB0aGUgbGFyZ2VzdCB2b2xjYW5vIChPbHltcHVzIE1vbnMsIDIxLjkga20gaGlnaCkuCgojIyMgT3V0ZXIgUGxhbmV0cyAoR2FzL0ljZSBHaWFudHMpCjUuICoqSnVwaXRlcioqOiBMYXJnZXN0IHBsYW5ldCAoMTF4IEVhcnRoJ3MgZGlhbWV0ZXIpLCBHcmVhdCBSZWQgU3BvdCBzdG9ybS4KNi4gKipTYXR1cm4qKjogRmFtb3VzIGZvciBpdHMgcmluZyBzeXN0ZW0sIDgzIGtub3duIG1vb25zIGluY2x1ZGluZyBUaXRhbi4KNy4gKipVcmFudXMqKjogUm90YXRlcyBvbiBpdHMgc2lkZSAoOTjCsCBheGlhbCB0aWx0KSwgMjcgbW9vbnMuCjguICoqTmVwdHVuZSoqOiBTdHJvbmdlc3Qgd2luZHMgaW4gdGhlIFNvbGFyIFN5c3RlbSAoMiwxMDAga20vaCkuCgojIyBOb3RhYmxlIE1pc3Npb25zCi0gKipWb3lhZ2VyIDEgJiAyICgxOTc3KSoqOiBGaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZS4KLSAqKkh1YmJsZSBTcGFjZSBUZWxlc2NvcGUgKDE5OTApKio6IFJldm9sdXRpb25pemVkIG91ciB2aWV3IG9mIHRoZSB1bml2ZXJzZS4KLSAqKk1hcnMgUm92ZXJzKio6IFNwaXJpdCwgT3Bwb3J0dW5pdHksIEN1cmlvc2l0eSwgUGVyc2V2ZXJhbmNlIGV4cGxvcmVkIE1hcnMuCi0gKipKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSAoMjAyMSkqKjogTW9zdCBwb3dlcmZ1bCBzcGFjZSB0ZWxlc2NvcGUgZXZlciBidWlsdC4KCiMjIER3YXJmIFBsYW5ldHMKUGx1dG8gd2FzIHJlY2xhc3NpZmllZCBhcyBhIGR3YXJmIHBsYW5ldCBpbiAyMDA2IGJ5IHRoZSBJbnRlcm5hdGlvbmFsIEFzdHJvbm9taWNhbCBVbmlvbi4gCk90aGVyIGR3YXJmIHBsYW5ldHMgaW5jbHVkZSBFcmlzLCBIYXVtZWEsIE1ha2VtYWtlLCBhbmQgQ2VyZXMuCgojIyBGdXR1cmUgb2YgU3BhY2UgRXhwbG9yYXRpb24KLSAqKjIwMjQtMjAyNSoqOiBBcnRlbWlzIHByb2dyYW0gYWltcyB0byByZXR1cm4gaHVtYW5zIHRvIHRoZSBNb29uLgotICoqMjAzMHMqKjogTkFTQSBhbmQgU3BhY2VYIHBsYW5uaW5nIGNyZXdlZCBtaXNzaW9ucyB0byBNYXJzLgotICoqTG9uZy10ZXJtKio6IFBvdGVudGlhbCBjb2xvbml6YXRpb24gb2YgTWFycyBhbmQgYXN0ZXJvaWQgbWluaW5nLgoiIiIKCiMgUXVlc3Rpb25zIGZvciB0ZXN0aW5nIC0gTWl4ZWQgZmFjdHVhbCBhbmQgcmVhc29uaW5nIHF1ZXN0aW9ucwojIEZhY3R1YWwgcXVlc3Rpb25zICh3aGF0L3doZW4vd2hlcmUpIHNob3VsZCB0cmlnZ2VyIEhZQlJJRCBtb2RlCiMgUmVhc29uaW5nIHF1ZXN0aW9ucyAod2h5L2V4cGxhaW4vZGVzY3JpYmUpIHNob3VsZCB0cmlnZ2VyIFRJVEFOU19PTkxZIG1vZGUKVEVTVF9RVUVTVElPTlMgPSB7CiAgICAiY2xpbWF0ZSI6IFsKICAgICAgICAjIEZhY3R1YWwgcXVlc3Rpb25zIChleHBlY3QgSFlCUklEKQogICAgICAgICgiV2hhdCBpcyB0aGUgY3VycmVudCBDTzIgY29uY2VudHJhdGlvbiBpbiB0aGUgYXRtb3NwaGVyZT8iLCAiNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMiKSwKICAgICAgICAoIkhvdyBtdWNoIGhhcyB0aGUgZ2xvYmFsIHRlbXBlcmF0dXJlIHJpc2VuIHNpbmNlIHByZS1pbmR1c3RyaWFsIHRpbWVzPyIsICJhcHByb3hpbWF0ZWx5IDEuMcKwQyIpLAogICAgICAgICgiV2hhdCB5ZWFyIHdhcyB0aGUgUGFyaXMgQWdyZWVtZW50IHNpZ25lZD8iLCAiMjAxNSIpLAogICAgICAgICgiSG93IG11Y2ggaGF2ZSBzZWEgbGV2ZWxzIHJpc2VuIHNpbmNlIDE5MDA/IiwgImFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBjbGltYXRlIGNoYW5nZSBwcmltYXJpbHkgY2F1c2VkIGJ5IGh1bWFuIGFjdGl2aXRpZXM/IiwgImJ1cm5pbmcgb2YgZm9zc2lsIGZ1ZWxzIHJlbGVhc2VzIGdyZWVuaG91c2UgZ2FzZXMgdGhhdCB0cmFwIGhlYXQiKSwKICAgICAgICAoIkV4cGxhaW4gaG93IHRoZSBncmVlbmhvdXNlIGVmZmVjdCB3b3JrcyIsICJncmVlbmhvdXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIGNyZWF0aW5nIHdhcm1pbmcgZWZmZWN0IiksCiAgICAgICAgKCJEZXNjcmliZSB0aGUgbWFpbiBpbXBhY3RzIG9mIGNsaW1hdGUgY2hhbmdlIG9uIGVjb3N5c3RlbXMiLCAiY29yYWwgYmxlYWNoaW5nLCBzcGVjaWVzIG1pZ3JhdGlvbiwgYmlvZGl2ZXJzaXR5IGxvc3MsIGFuZCBoYWJpdGF0IGRpc3J1cHRpb24iKSwKICAgIF0sCiAgICAiYWkiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIldobyBjb2luZWQgdGhlIHRlcm0gJ0FydGlmaWNpYWwgSW50ZWxsaWdlbmNlJz8iLCAiSm9obiBNY0NhcnRoeSBpbiAxOTU2IiksCiAgICAgICAgKCJXaGVuIGRpZCBEZWVwIEJsdWUgZGVmZWF0IEdhcnJ5IEthc3Bhcm92PyIsICIxOTk3IiksCiAgICAgICAgKCJXaGF0IGlzIHRoZSBlc3RpbWF0ZWQgZWNvbm9taWMgaW1wYWN0IG9mIEFJIGJ5IDIwMzA/IiwgIiQxNS43IHRyaWxsaW9uIGFjY29yZGluZyB0byBQd0MiKSwKICAgICAgICAoIldoYXQgYXJjaGl0ZWN0dXJlIHBvd2VycyBHUFQtNCBhbmQgQ2xhdWRlPyIsICJUcmFuc2Zvcm1lcnMiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBiaWFzIGluIEFJIGEgbWFqb3IgZXRoaWNhbCBjb25jZXJuPyIsICJiaWFzIGluIHRyYWluaW5nIGRhdGEgY2FuIGxlYWQgdG8gdW5mYWlyIG91dGNvbWVzIGFuZCBkaXNjcmltaW5hdGlvbiIpLAogICAgICAgICgiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIE5hcnJvdyBBSSBhbmQgR2VuZXJhbCBBSSIsICJOYXJyb3cgQUkgaXMgZGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIHdoaWxlIEdlbmVyYWwgQUkgaGFzIGh1bWFuLWxldmVsIGNvZ25pdGl2ZSBhYmlsaXRpZXMgYWNyb3NzIGFsbCBkb21haW5zIiksCiAgICAgICAgKCJEZXNjcmliZSBob3cgcmVpbmZvcmNlbWVudCBsZWFybmluZyB3b3JrcyIsICJ0cmFpbmluZyBhZ2VudHMgdGhyb3VnaCByZXdhcmQtYmFzZWQgZmVlZGJhY2sgdG8gbGVhcm4gb3B0aW1hbCBiZWhhdmlvcnMiKSwKICAgIF0sCiAgICAic3BhY2UiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIkhvdyBvbGQgaXMgdGhlIFNvbGFyIFN5c3RlbT8iLCAiYXBwcm94aW1hdGVseSA0LjYgYmlsbGlvbiB5ZWFycyIpLAogICAgICAgICgiV2hhdCBpcyB0aGUgdGFsbGVzdCB2b2xjYW5vIGluIHRoZSBTb2xhciBTeXN0ZW0/IiwgIk9seW1wdXMgTW9ucywgMjEuOSBrbSBoaWdoIiksCiAgICAgICAgKCJXaGVuIHdhcyB0aGUgSmFtZXMgV2ViYiBTcGFjZSBUZWxlc2NvcGUgbGF1bmNoZWQ/IiwgIjIwMjEiKSwKICAgICAgICAoIldoYXQgeWVhciB3YXMgUGx1dG8gcmVjbGFzc2lmaWVkIGFzIGEgZHdhcmYgcGxhbmV0PyIsICIyMDA2IiksCiAgICAgICAgCiAgICAgICAgIyBSZWFzb25pbmcgcXVlc3Rpb25zIChleHBlY3QgVElUQU5TX09OTFkpCiAgICAgICAgKCJXaHkgaXMgVmVudXMgdGhlIGhvdHRlc3QgcGxhbmV0IGRlc3BpdGUgTWVyY3VyeSBiZWluZyBjbG9zZXIgdG8gdGhlIFN1bj8iLCAiVmVudXMgaGFzIGEgdGhpY2sgYXRtb3NwaGVyZSB0aGF0IHRyYXBzIGhlYXQgdGhyb3VnaCBncmVlbmhvdXNlIGVmZmVjdCIpLAogICAgICAgICgiRXhwbGFpbiB3aHkgb3V0ZXIgcGxhbmV0cyBhcmUgY2FsbGVkIGdhcyBnaWFudHMiLCAidGhleSBhcmUgbXVjaCBsYXJnZXIgYW5kIGNvbXBvc2VkIHByaW1hcmlseSBvZiBnYXNlcyBsaWtlIGh5ZHJvZ2VuIGFuZCBoZWxpdW0gcmF0aGVyIHRoYW4gc29saWQgcm9jayIpLAogICAgICAgICgiRGVzY3JpYmUgdGhlIHNpZ25pZmljYW5jZSBvZiB0aGUgVm95YWdlciBtaXNzaW9ucyIsICJmaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZSBhbmQgZXhwbG9yZSB0aGUgb3V0ZXIgc29sYXIgc3lzdGVtIiksCiAgICBdLAp9CgpkZWYgZ2V0X2FsbF9lc3NheXMoKToKICAgICIiIlJldHVybiBhbGwgZXNzYXlzIGFzIGEgZGljdGlvbmFyeS4iIiIKICAgIHJldHVybiB7CiAgICAgICAgImNsaW1hdGUiOiBFU1NBWV9DTElNQVRFLAogICAgICAgICJhaSI6IEVTU0FZX0FJLAogICAgICAgICJzcGFjZSI6IEVTU0FZX1NQQUNFLAogICAgfQoKZGVmIGdldF90ZXN0X3F1ZXN0aW9ucygpOgogICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiIKICAgIHJldHVybiBURVNUX1FVRVNUSU9OUwo=',
    'projects/hybrid_titans/common/__init__.py': 'IyBDb21tb24gbW9kdWxlcw==',
    'projects/hybrid_titans/common/embedders.py': 'IiIiClNlbWFudGljIEVtYmVkZGVyIHVzaW5nIHNlbnRlbmNlLXRyYW5zZm9ybWVycy4KU2hhcmVkIGJ5IGFsbCBkZW1vIHNjcmlwdHMuCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCmNsYXNzIFNlbnRlbmNlVHJhbnNmb3JtZXJFbWJlZGRlcihubi5Nb2R1bGUpOgogICAgIiIiU2VtYW50aWMgZW1iZWRkZXIgdXNpbmcgc2VudGVuY2UtdHJhbnNmb3JtZXJzLiIiIgogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxfbmFtZT0iYWxsLU1pbmlMTS1MNi12MiIsIHRhcmdldF9kaW09MjU2KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgogICAgICAgIHNlbGYubW9kZWwgPSBTZW50ZW5jZVRyYW5zZm9ybWVyKG1vZGVsX25hbWUpCiAgICAgICAgc2VsZi5zb3VyY2VfZGltID0gc2VsZi5tb2RlbC5nZXRfc2VudGVuY2VfZW1iZWRkaW5nX2RpbWVuc2lvbigpCiAgICAgICAgc2VsZi50YXJnZXRfZGltID0gdGFyZ2V0X2RpbQogICAgICAgIAogICAgICAgIGlmIHNlbGYuc291cmNlX2RpbSAhPSB0YXJnZXRfZGltOgogICAgICAgICAgICBzZWxmLnByb2plY3Rpb24gPSBubi5MaW5lYXIoc2VsZi5zb3VyY2VfZGltLCB0YXJnZXRfZGltKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYucHJvamVjdGlvbiA9IE5vbmUKICAgICAgICAKICAgICAgICBwcmludChmIuKchSBMb2FkZWQgU2VudGVuY2VUcmFuc2Zvcm1lcjoge21vZGVsX25hbWV9IChkaW09e3NlbGYuc291cmNlX2RpbX0pIikKICAgIAogICAgZGVmIGZvcndhcmQoc2VsZiwgdGV4dCk6CiAgICAgICAgIiIiRW1iZWQgdGV4dCBpbnRvIGEgZml4ZWQtc2l6ZSB2ZWN0b3IuIiIiCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IHNlbGYubW9kZWwuZW5jb2RlKHRleHQsIGNvbnZlcnRfdG9fdGVuc29yPVRydWUpCiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy5jbG9uZSgpLmNwdSgpCiAgICAgICAgCiAgICAgICAgaWYgc2VsZi5wcm9qZWN0aW9uIGlzIG5vdCBOb25lOgogICAgICAgICAgICBlbWJlZGRpbmcgPSBzZWxmLnByb2plY3Rpb24oZW1iZWRkaW5nLmZsb2F0KCkpCiAgICAgICAgCiAgICAgICAgIyBFbnN1cmUgM0Qgb3V0cHV0OiBbMSwgc2VxX2xlbiwgZGltXQogICAgICAgIGlmIGVtYmVkZGluZy5kaW0oKSA9PSAxOgogICAgICAgICAgICBlbWJlZGRpbmcgPSBlbWJlZGRpbmcudW5zcXVlZXplKDApLnVuc3F1ZWV6ZSgwKQogICAgICAgIGVsaWYgZW1iZWRkaW5nLmRpbSgpID09IDI6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy51bnNxdWVlemUoMCkKICAgICAgICAgICAgCiAgICAgICAgcmV0dXJuIGVtYmVkZGluZwogICAgCiAgICBkZWYgZW1iZWRfYmF0Y2goc2VsZiwgdGV4dHMpOgogICAgICAgICIiIkVtYmVkIGEgYmF0Y2ggb2YgdGV4dHMuIiIiCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGVtYmVkZGluZ3MgPSBzZWxmLm1vZGVsLmVuY29kZSh0ZXh0cywgY29udmVydF90b190ZW5zb3I9VHJ1ZSkKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IGVtYmVkZGluZ3MuY2xvbmUoKS5jcHUoKQogICAgICAgIAogICAgICAgIGlmIHNlbGYucHJvamVjdGlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmdzLmZsb2F0KCkpCiAgICAgICAgCiAgICAgICAgcmV0dXJuIGVtYmVkZGluZ3MK',
    'projects/hybrid_titans/common/titan_utils.py': 'IiIiClRpdGFuIG1vZGVsIHV0aWxpdGllcy4KU2hhcmVkIGluaXRpYWxpemF0aW9uIGFuZCBjb25maWd1cmF0aW9uIGZvciBUaXRhbk1BRy9UaXRhblJBRy4KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHN5cwppbXBvcnQgb3MKCiMgQWRkIHBhcmVudCBwYXRocyBmb3IgaW1wb3J0cwpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpKQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKSwgJ3NyYycpKQoKZnJvbSBtYWluIGltcG9ydCBUaXRhbk1BRywgVGl0YW5NQUMsIFRpdGFuUkFHCgoKZGVmIGNyZWF0ZV90aXRhbl9yYWcoZGltPTI1NiwgYXJjaD0iTUFHIiwgd2luZG93X3NpemU9MzIsIG1lbW9yeV9kZXB0aD0zLCAKICAgICAgICAgICAgICAgICAgICAgbnVtX3BlcnNpc3RlbnRfdG9rZW5zPTQsIHRocmVzaG9sZD0wLjAsCiAgICAgICAgICAgICAgICAgICAgIGxlYXJuaW5nX3JhdGU9MC4wMSwgZm9yZ2V0dGluZ19yYXRlPTAuMDAwMSk6CiAgICAiIiIKICAgIENyZWF0ZSBhbmQgY29uZmlndXJlIGEgVGl0YW5SQUcgaW5zdGFuY2UuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgZGltOiBFbWJlZGRpbmcgZGltZW5zaW9uCiAgICAgICAgYXJjaDogQXJjaGl0ZWN0dXJlIHR5cGUgKCJNQUciIG9yICJNQUMiKQogICAgICAgIHdpbmRvd19zaXplOiBTbGlkaW5nIHdpbmRvdyBzaXplIGZvciBhdHRlbnRpb24KICAgICAgICBtZW1vcnlfZGVwdGg6IE51bWJlciBvZiBsYXllcnMgaW4gbWVtb3J5IE1MUAogICAgICAgIG51bV9wZXJzaXN0ZW50X3Rva2VuczogTnVtYmVyIG9mIHBlcnNpc3RlbnQgbWVtb3J5IHRva2VucwogICAgICAgIHRocmVzaG9sZDogU3VycHJpc2UgdGhyZXNob2xkIGZvciBzcGFyc2UgdXBkYXRlcwogICAgICAgIGxlYXJuaW5nX3JhdGU6IExlYXJuaW5nIHJhdGUgKHRoZXRhKSBmb3IgbWVtb3J5IHVwZGF0ZXMKICAgICAgICBmb3JnZXR0aW5nX3JhdGU6IEZvcmdldHRpbmcgcmF0ZSAoYWxwaGEpIGZvciBtZW1vcnkgZGVjYXkKICAgIAogICAgUmV0dXJuczoKICAgICAgICBUaXRhblJBRzogQ29uZmlndXJlZCBUaXRhblJBRyBpbnN0YW5jZQogICAgIiIiCiAgICBpZiBhcmNoLnVwcGVyKCkgPT0gIk1BQyI6CiAgICAgICAgYmFzZV9tb2RlbCA9IFRpdGFuTUFDKAogICAgICAgICAgICBkaW09ZGltLAogICAgICAgICAgICBjaHVua19zaXplPXdpbmRvd19zaXplLAogICAgICAgICAgICBoaWRkZW5fZGltPWRpbSAqIDIsCiAgICAgICAgICAgIG1lbW9yeV9kZXB0aD1tZW1vcnlfZGVwdGgsCiAgICAgICAgICAgIG51bV9wZXJzaXN0ZW50X3Rva2Vucz1udW1fcGVyc2lzdGVudF90b2tlbnMsCiAgICAgICAgICAgIHRocmVzaG9sZD10aHJlc2hvbGQKICAgICAgICApCiAgICBlbHNlOgogICAgICAgIGJhc2VfbW9kZWwgPSBUaXRhbk1BRygKICAgICAgICAgICAgZGltPWRpbSwKICAgICAgICAgICAgd2luZG93X3NpemU9d2luZG93X3NpemUsCiAgICAgICAgICAgIGhpZGRlbl9kaW09ZGltICogMiwKICAgICAgICAgICAgbWVtb3J5X2RlcHRoPW1lbW9yeV9kZXB0aCwKICAgICAgICAgICAgbnVtX3BlcnNpc3RlbnRfdG9rZW5zPW51bV9wZXJzaXN0ZW50X3Rva2VucywKICAgICAgICAgICAgdGhyZXNob2xkPXRocmVzaG9sZAogICAgICAgICkKICAgIAogICAgIyBDb25maWd1cmUgbWVtb3J5IHBhcmFtZXRlcnMKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGJhc2VfbW9kZWwubHRtLnRoZXRhLmZpbGxfKGxlYXJuaW5nX3JhdGUpCiAgICAgICAgYmFzZV9tb2RlbC5sdG0uYWxwaGEuZmlsbF8oZm9yZ2V0dGluZ19yYXRlKQogICAgCiAgICB0aXRhbl9yYWcgPSBUaXRhblJBRyhiYXNlX21vZGVsKQogICAgCiAgICBwcmludChmIuKchSBDcmVhdGVkIFRpdGFuUkFHIHdpdGgge2FyY2h9IGFyY2hpdGVjdHVyZSAoZGltPXtkaW19KSIpCiAgICAKICAgIHJldHVybiB0aXRhbl9yYWcKCgpkZWYgZGlnZXN0X2RvY3VtZW50KHRpdGFuX3JhZywgZW1iZWRkZXIsIHRleHQsIGVwb2Nocz01MCk6CiAgICAiIiIKICAgIERpZ2VzdCBhIGRvY3VtZW50IGludG8gVGl0YW5SQUcgbWVtb3J5LgogICAgCiAgICBBcmdzOgogICAgICAgIHRpdGFuX3JhZzogVGl0YW5SQUcgaW5zdGFuY2UKICAgICAgICBlbWJlZGRlcjogRW1iZWRkZXIgdG8gY29udmVydCB0ZXh0IHRvIGVtYmVkZGluZ3MKICAgICAgICB0ZXh0OiBEb2N1bWVudCB0ZXh0ICh3aWxsIGJlIHNwbGl0IGJ5IHBhcmFncmFwaHMpCiAgICAgICAgZXBvY2hzOiBOdW1iZXIgb2YgZGlnZXN0aW9uIGVwb2NocwogICAgCiAgICBSZXR1cm5zOgogICAgICAgIHR1cGxlOiAocGFyYWdyYXBocywgcGFyYWdyYXBoX2VtYmVkZGluZ3MpCiAgICAiIiIKICAgICMgUGFyc2UgaW50byBwYXJhZ3JhcGhzCiAgICByYXdfcGFyYWdyYXBocyA9IHRleHQuc3BsaXQoJ1xuXG4nKQogICAgcGFyYWdyYXBocyA9IFtdCiAgICBmb3IgcGFyYSBpbiByYXdfcGFyYWdyYXBoczoKICAgICAgICBjbGVhbmVkID0gJyAnLmpvaW4obGluZS5zdHJpcCgpIGZvciBsaW5lIGluIHBhcmEuc3BsaXQoJ1xuJykgaWYgbGluZS5zdHJpcCgpKQogICAgICAgIGlmIGNsZWFuZWQgYW5kIGxlbihjbGVhbmVkKSA+IDEwOgogICAgICAgICAgICBwYXJhZ3JhcGhzLmFwcGVuZChjbGVhbmVkKQogICAgCiAgICAjIEVtYmVkIHBhcmFncmFwaHMKICAgIGFsbF9lbWJlZGRpbmdzID0gdG9yY2guY2F0KFtlbWJlZGRlcihwKSBmb3IgcCBpbiBwYXJhZ3JhcGhzXSwgZGltPTEpCiAgICAKICAgICMgRGlnZXN0IGludG8gbWVtb3J5CiAgICBwcmludChmIvCfp6AgRGlnZXN0aW5nIHtsZW4ocGFyYWdyYXBocyl9IHBhcmFncmFwaHMgKHtlcG9jaHN9IGVwb2NocykuLi4iKQogICAgZm9yIGVwb2NoIGluIHJhbmdlKGVwb2Nocyk6CiAgICAgICAgdGl0YW5fcmFnLmRpZ2VzdF9rbm93bGVkZ2UoYWxsX2VtYmVkZGluZ3MpCiAgICAgICAgaWYgKGVwb2NoICsgMSkgJSAxMCA9PSAwOgogICAgICAgICAgICBwcmludChmIiAgIEVwb2NoIHtlcG9jaCArIDF9L3tlcG9jaHN9IGNvbXBsZXRlZCIpCiAgICAKICAgICMgQ29tcHV0ZSBwYXJhZ3JhcGggZW1iZWRkaW5ncyBmb3IgcmV0cmlldmFsCiAgICBwYXJhZ3JhcGhfZW1iZWRkaW5ncyA9IGVtYmVkZGVyLmVtYmVkX2JhdGNoKHBhcmFncmFwaHMpCiAgICAKICAgIHJldHVybiBwYXJhZ3JhcGhzLCBwYXJhZ3JhcGhfZW1iZWRkaW5ncwo=',
    'projects/hybrid_titans/common/llm_utils.py': 'IiIiCkxMTSB1dGlsaXRpZXMgZm9yIHRleHQgZ2VuZXJhdGlvbi4KU2hhcmVkIEZsYW4tVDUgbG9hZGluZyBhbmQgZ2VuZXJhdGlvbiBmdW5jdGlvbnMuCiIiIgoKaW1wb3J0IHRvcmNoCgoKY2xhc3MgRmxhblQ1R2VuZXJhdG9yOgogICAgIiIiRmxhbi1UNSBiYXNlZCB0ZXh0IGdlbmVyYXRvci4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsX25hbWU9Imdvb2dsZS9mbGFuLXQ1LWxhcmdlIik6CiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IFQ1Rm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uLCBUNVRva2VuaXplcgogICAgICAgIAogICAgICAgIHNlbGYudG9rZW5pemVyID0gVDVUb2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUpCiAgICAgICAgc2VsZi5sbG0gPSBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbi5mcm9tX3ByZXRyYWluZWQobW9kZWxfbmFtZSkKICAgICAgICBzZWxmLm1vZGVsX25hbWUgPSBtb2RlbF9uYW1lCiAgICAgICAgCiAgICAgICAgcHJpbnQoZiLinIUgTG9hZGVkIExMTToge21vZGVsX25hbWV9IikKICAgIAogICAgZGVmIGdlbmVyYXRlKHNlbGYsIHByb21wdCwgbWF4X25ld190b2tlbnM9MTAwLCBudW1fYmVhbXM9NCk6CiAgICAgICAgIiIiCiAgICAgICAgR2VuZXJhdGUgdGV4dCBmcm9tIGEgcHJvbXB0LgogICAgICAgIAogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHByb21wdDogSW5wdXQgcHJvbXB0CiAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zOiBNYXhpbXVtIHRva2VucyB0byBnZW5lcmF0ZQogICAgICAgICAgICBudW1fYmVhbXM6IEJlYW0gc2VhcmNoIHdpZHRoCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgc3RyOiBHZW5lcmF0ZWQgdGV4dAogICAgICAgICIiIgogICAgICAgIGlucHV0cyA9IHNlbGYudG9rZW5pemVyKAogICAgICAgICAgICBwcm9tcHQsCiAgICAgICAgICAgIHJldHVybl90ZW5zb3JzPSJwdCIsCiAgICAgICAgICAgIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICAgICAgbWF4X2xlbmd0aD03NjgKICAgICAgICApCiAgICAgICAgCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIG91dHB1dHMgPSBzZWxmLmxsbS5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICBudW1fYmVhbXM9bnVtX2JlYW1zLAogICAgICAgICAgICAgICAgZWFybHlfc3RvcHBpbmc9VHJ1ZSwKICAgICAgICAgICAgICAgIG5vX3JlcGVhdF9uZ3JhbV9zaXplPTIKICAgICAgICAgICAgKQogICAgICAgIAogICAgICAgIGFuc3dlciA9IHNlbGYudG9rZW5pemVyLmRlY29kZShvdXRwdXRzWzBdLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpCiAgICAgICAgcmV0dXJuIGFuc3dlcgogICAgCiAgICBkZWYgZ2VuZXJhdGVfcWEoc2VsZiwgcXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zPTEwMCk6CiAgICAgICAgIiIiCiAgICAgICAgR2VuZXJhdGUgYW5zd2VyIGZvciBhIHF1ZXN0aW9uIGdpdmVuIGNvbnRleHQuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcXVlc3Rpb246IFRoZSBxdWVzdGlvbgogICAgICAgICAgICBjb250ZXh0OiBDb250ZXh0L3Bhc3NhZ2UgdG8gYW5zd2VyIGZyb20KICAgICAgICAgICAgbWF4X25ld190b2tlbnM6IE1heGltdW0gdG9rZW5zIHRvIGdlbmVyYXRlCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgdHVwbGU6IChhbnN3ZXIsIHByb21wdCkKICAgICAgICAiIiIKICAgICAgICBwcm9tcHQgPSBmIiIiQW5zd2VyIHRoZSBxdWVzdGlvbiBiYXNlZCBvbiB0aGUgY29udGV4dCBwcm92aWRlZC4KRXh0cmFjdCBzcGVjaWZpYyBkZXRhaWxzOiBleGFjdCBudW1iZXJzLCBuYW1lcywgZGF0ZXMsIGFuZCB0ZWNobmljYWwgdGVybXMuClByb3ZpZGUgdGhlIG1vc3QgcmVsZXZhbnQgYW5zd2VyIGZyb20gdGhlIGNvbnRleHQsIGV2ZW4gaWYgcGFydGlhbC4KCkNvbnRleHQ6Cntjb250ZXh0fQoKUXVlc3Rpb246IHtxdWVzdGlvbn0KCkFuc3dlcjoiIiIKICAgICAgICAKICAgICAgICBhbnN3ZXIgPSBzZWxmLmdlbmVyYXRlKHByb21wdCwgbWF4X25ld190b2tlbnMpCiAgICAgICAgcmV0dXJuIGFuc3dlciwgcHJvbXB0Cg==',
    'projects/hybrid_titans/common/text_utils.py': 'IiIiClRleHQgdXRpbGl0aWVzIGZvciBkb2N1bWVudCBwcm9jZXNzaW5nLgpTaGFyZWQgY2h1bmtpbmcgYW5kIHRleHQgcHJvY2Vzc2luZyBmdW5jdGlvbnMuCiIiIgoKaW1wb3J0IHJlCgoKZGVmIHNwbGl0X2ludG9fY2h1bmtzKHRleHQsIG1pbl9sZW5ndGg9MzAsIHNlbnRlbmNlc19wZXJfY2h1bms9Mywgb3ZlcmxhcF9zZW50ZW5jZXM9MSk6CiAgICAiIiIKICAgIFNwbGl0IHRleHQgaW50byBzZW1hbnRpYyBjaHVua3MgZm9yIHJldHJpZXZhbCB3aXRoIG92ZXJsYXBwaW5nIHdpbmRvd3MuCiAgICAKICAgIFN0cmF0ZWd5OgogICAgLSBIZWFkZXJzIGFyZSBncm91cGVkIHdpdGggdGhlaXIgZm9sbG93aW5nIHNlbnRlbmNlcwogICAgLSBDaHVua3Mgb3ZlcmxhcCBieSAxIHNlbnRlbmNlIHRvIGF2b2lkIGJvdW5kYXJ5IGluZm9ybWF0aW9uIGxvc3MKICAgIC0gRWFjaCBjaHVuayBjb250YWlucyAzIHNlbnRlbmNlcyBmb3IgYmV0dGVyIGNvbnRleHQKICAgIAogICAgQXJnczoKICAgICAgICB0ZXh0OiBJbnB1dCB0ZXh0IChjYW4gY29udGFpbiBtYXJrZG93biBoZWFkZXJzKQogICAgICAgIG1pbl9sZW5ndGg6IE1pbmltdW0gY2h1bmsgbGVuZ3RoIHRvIGtlZXAKICAgICAgICBzZW50ZW5jZXNfcGVyX2NodW5rOiBOdW1iZXIgb2Ygc2VudGVuY2VzIHBlciBjaHVuayAoZGVmYXVsdDogMykKICAgICAgICBvdmVybGFwX3NlbnRlbmNlczogTnVtYmVyIG9mIHNlbnRlbmNlcyB0byBvdmVybGFwIGJldHdlZW4gY2h1bmtzIChkZWZhdWx0OiAxKQogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBMaXN0IG9mIHRleHQgY2h1bmtzCiAgICAiIiIKICAgIGxpbmVzID0gdGV4dC5zcGxpdCgnXG4nKQogICAgY2h1bmtzID0gW10KICAgIGN1cnJlbnRfaGVhZGVyID0gIiIKICAgIGFsbF9zZW50ZW5jZXMgPSBbXSAgIyBDb2xsZWN0IGFsbCBzZW50ZW5jZXMgZmlyc3QKICAgIAogICAgIyBGaXJzdCBwYXNzOiBjb2xsZWN0IGFsbCBzZW50ZW5jZXMgd2l0aCB0aGVpciBoZWFkZXJzCiAgICBmb3IgbGluZSBpbiBsaW5lczoKICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgCiAgICAgICAgIyBDaGVjayBpZiBpdCdzIGEgaGVhZGVyCiAgICAgICAgaWYgbGluZS5zdGFydHN3aXRoKCcjJyk6CiAgICAgICAgICAgICMgSWYgd2UgaGF2ZSBhY2N1bXVsYXRlZCBzZW50ZW5jZXMsIGNyZWF0ZSBjaHVuayBiZWZvcmUgbmV3IGhlYWRlcgogICAgICAgICAgICBpZiBhbGxfc2VudGVuY2VzOgogICAgICAgICAgICAgICAgX2NyZWF0ZV9jaHVua3NfZnJvbV9zZW50ZW5jZXMoCiAgICAgICAgICAgICAgICAgICAgY2h1bmtzLCBjdXJyZW50X2hlYWRlciwgYWxsX3NlbnRlbmNlcywgCiAgICAgICAgICAgICAgICAgICAgc2VudGVuY2VzX3Blcl9jaHVuaywgb3ZlcmxhcF9zZW50ZW5jZXMsIG1pbl9sZW5ndGgKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGFsbF9zZW50ZW5jZXMgPSBbXQogICAgICAgICAgICBjdXJyZW50X2hlYWRlciA9IGxpbmUKICAgICAgICAgICAgY29udGludWUKICAgICAgICAKICAgICAgICAjIEhhbmRsZSByZWd1bGFyIGNvbnRlbnQgLSBzcGxpdCBpbnRvIHNlbnRlbmNlcwogICAgICAgICMgUHJvdGVjdCBhYmJyZXZpYXRpb25zCiAgICAgICAgcHJvdGVjdGVkID0gbGluZS5yZXBsYWNlKCdEci4nLCAnRHLCpycpLnJlcGxhY2UoJ01yLicsICdNcsKnJykucmVwbGFjZSgnTXMuJywgJ01zwqcnKQogICAgICAgIHByb3RlY3RlZCA9IHByb3RlY3RlZC5yZXBsYWNlKCdldGMuJywgJ2V0Y8KnJykucmVwbGFjZSgnZS5nLicsICdlZ8KnJykucmVwbGFjZSgnaS5lLicsICdpZcKnJykKICAgICAgICAKICAgICAgICAjIFNwbGl0IG9uIHNlbnRlbmNlIGJvdW5kYXJpZXMKICAgICAgICBwYXJ0cyA9IHJlLnNwbGl0KHInKD88PVsuIT9dKVxzKycsIHByb3RlY3RlZCkKICAgICAgICAKICAgICAgICBmb3IgcGFydCBpbiBwYXJ0czoKICAgICAgICAgICAgcGFydCA9IHBhcnQuc3RyaXAoKQogICAgICAgICAgICAjIFJlc3RvcmUgYWJicmV2aWF0aW9ucwogICAgICAgICAgICBwYXJ0ID0gcGFydC5yZXBsYWNlKCdEcsKnJywgJ0RyLicpLnJlcGxhY2UoJ01ywqcnLCAnTXIuJykucmVwbGFjZSgnTXPCpycsICdNcy4nKQogICAgICAgICAgICBwYXJ0ID0gcGFydC5yZXBsYWNlKCdldGPCpycsICdldGMuJykucmVwbGFjZSgnZWfCpycsICdlLmcuJykucmVwbGFjZSgnaWXCpycsICdpLmUuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGlmIGxlbihwYXJ0KSA+PSAxMDogICMgTWluaW11bSBzZW50ZW5jZSBsZW5ndGgKICAgICAgICAgICAgICAgIGFsbF9zZW50ZW5jZXMuYXBwZW5kKHBhcnQpCiAgICAKICAgICMgRmx1c2ggcmVtYWluaW5nIHNlbnRlbmNlcwogICAgaWYgYWxsX3NlbnRlbmNlcyBvciBjdXJyZW50X2hlYWRlcjoKICAgICAgICBfY3JlYXRlX2NodW5rc19mcm9tX3NlbnRlbmNlcygKICAgICAgICAgICAgY2h1bmtzLCBjdXJyZW50X2hlYWRlciwgYWxsX3NlbnRlbmNlcywgCiAgICAgICAgICAgIHNlbnRlbmNlc19wZXJfY2h1bmssIG92ZXJsYXBfc2VudGVuY2VzLCBtaW5fbGVuZ3RoCiAgICAgICAgKQogICAgCiAgICByZXR1cm4gY2h1bmtzCgoKZGVmIF9jcmVhdGVfY2h1bmtzX2Zyb21fc2VudGVuY2VzKGNodW5rcywgaGVhZGVyLCBzZW50ZW5jZXMsIAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlbnRlbmNlc19wZXJfY2h1bmssIG92ZXJsYXBfc2VudGVuY2VzLCBtaW5fbGVuZ3RoKToKICAgICIiIkhlbHBlcjogQ3JlYXRlIG92ZXJsYXBwaW5nIGNodW5rcyBmcm9tIGEgbGlzdCBvZiBzZW50ZW5jZXMuIiIiCiAgICBpZiBub3Qgc2VudGVuY2VzOgogICAgICAgIGlmIGhlYWRlciBhbmQgbGVuKGhlYWRlcikgPiA1OgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGhlYWRlcikKICAgICAgICByZXR1cm4KICAgIAogICAgIyBJZiB3ZSBoYXZlIGZld2VyIHNlbnRlbmNlcyB0aGFuIGNodW5rIHNpemUsIGNyZWF0ZSBvbmUgY2h1bmsKICAgIGlmIGxlbihzZW50ZW5jZXMpIDw9IHNlbnRlbmNlc19wZXJfY2h1bms6CiAgICAgICAgY29udGVudCA9ICIgIi5qb2luKHNlbnRlbmNlcykKICAgICAgICBpZiBoZWFkZXI6CiAgICAgICAgICAgIGNodW5rID0gZiJ7aGVhZGVyfSB7Y29udGVudH0iCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2h1bmsgPSBjb250ZW50CiAgICAgICAgaWYgbGVuKGNodW5rKSA+PSBtaW5fbGVuZ3RoOgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGNodW5rKQogICAgICAgIHJldHVybgogICAgCiAgICAjIENyZWF0ZSBvdmVybGFwcGluZyBjaHVua3MgdXNpbmcgc2xpZGluZyB3aW5kb3cKICAgIHN0ZXAgPSBzZW50ZW5jZXNfcGVyX2NodW5rIC0gb3ZlcmxhcF9zZW50ZW5jZXMKICAgIHN0ZXAgPSBtYXgoMSwgc3RlcCkgICMgRW5zdXJlIGF0IGxlYXN0IDEgc3RlcAogICAgCiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oc2VudGVuY2VzKSwgc3RlcCk6CiAgICAgICAgd2luZG93ID0gc2VudGVuY2VzW2k6aSArIHNlbnRlbmNlc19wZXJfY2h1bmtdCiAgICAgICAgaWYgbm90IHdpbmRvdzoKICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgCiAgICAgICAgY29udGVudCA9ICIgIi5qb2luKHdpbmRvdykKICAgICAgICAKICAgICAgICAjIEluY2x1ZGUgaGVhZGVyIG9ubHkgaW4gZmlyc3QgY2h1bmsgb2Ygc2VjdGlvbgogICAgICAgIGlmIGkgPT0gMCBhbmQgaGVhZGVyOgogICAgICAgICAgICBjaHVuayA9IGYie2hlYWRlcn0ge2NvbnRlbnR9IgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNodW5rID0gY29udGVudAogICAgICAgICAgICAKICAgICAgICBpZiBsZW4oY2h1bmspID49IG1pbl9sZW5ndGg6CiAgICAgICAgICAgIGNodW5rcy5hcHBlbmQoY2h1bmspCiAgICAgICAgCiAgICAgICAgIyBTdG9wIGlmIHdlJ3ZlIHByb2Nlc3NlZCBhbGwgc2VudGVuY2VzCiAgICAgICAgaWYgaSArIHNlbnRlbmNlc19wZXJfY2h1bmsgPj0gbGVuKHNlbnRlbmNlcyk6CiAgICAgICAgICAgIGJyZWFrCgoKZGVmIHNpbXBsZV9zcGxpdCh0ZXh0KToKICAgICIiIlNpbXBsZSBsaW5lLWJ5LWxpbmUgc3BsaXQgKGZvciBjb21wYXJpc29uKS4iIiIKICAgIHJldHVybiBbbGluZS5zdHJpcCgpIGZvciBsaW5lIGluIHRleHQuc3BsaXQoJ1xuJykgaWYgbGluZS5zdHJpcCgpIGFuZCBsZW4obGluZS5zdHJpcCgpKSA+IDEwXQo=',
    'projects/hybrid_titans/baselines.py': 'IiIiCkJhc2VsaW5lIFJldHJpZXZhbCBTdHJhdGVnaWVzIGZvciBUaXRhblJBRyBldmFsdWF0aW9uLgoKVGhyZWUgbW9kZXM6Ci0gUHVyZVJBRzogQk0yNSArIEVtYmVkZGluZyAobm8gTWVtb3J5KQotIFRpdGFuT25seTogTWVtb3J5LWd1aWRlZCByZXRyaWV2YWwgb25seQotIEh5YnJpZFJBRzogQk0yNSArIE1lbW9yeSArIEVtYmVkZGluZyBmdXNpb24KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgVHVwbGUsIERpY3QsIE9wdGlvbmFsCgoKZGVmIG5vcm1hbGl6ZV9zY29yZXMoc2NvcmVzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIk5vcm1hbGl6ZSBzY29yZXMgdG8gWzAsIDFdIHJhbmdlLiIiIgogICAgbWluX3MsIG1heF9zID0gc2NvcmVzLm1pbigpLCBzY29yZXMubWF4KCkKICAgIGlmIG1heF9zIC0gbWluX3MgPCAxZS02OgogICAgICAgIHJldHVybiB0b3JjaC56ZXJvc19saWtlKHNjb3JlcykKICAgIHJldHVybiAoc2NvcmVzIC0gbWluX3MpIC8gKG1heF9zIC0gbWluX3MpCgoKZGVmIGNvbXB1dGVfYm0yNV9zY29yZXMocXVlc3Rpb246IHN0ciwgZG9jdW1lbnRzOiBMaXN0W3N0cl0pIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIgogICAgQ29tcHV0ZSBzaW1wbGUgQk0yNS1saWtlIGtleXdvcmQgbWF0Y2hpbmcgc2NvcmVzLgogICAgCiAgICBBcmdzOgogICAgICAgIHF1ZXN0aW9uOiBRdWVyeSB0ZXh0CiAgICAgICAgZG9jdW1lbnRzOiBMaXN0IG9mIGRvY3VtZW50IHN0cmluZ3MKICAgIAogICAgUmV0dXJuczoKICAgICAgICB0b3JjaC5UZW5zb3I6IEtleXdvcmQgbWF0Y2ggc2NvcmVzIGZvciBlYWNoIGRvY3VtZW50CiAgICAiIiIKICAgIHF1ZXN0aW9uX2xvd2VyID0gcXVlc3Rpb24ubG93ZXIoKQogICAgcXVlc3Rpb25fd29yZHMgPSBbdyBmb3IgdyBpbiBxdWVzdGlvbl9sb3dlci5zcGxpdCgpIGlmIGxlbih3KSA+IDNdCiAgICAKICAgIHNjb3JlcyA9IFtdCiAgICBmb3IgZG9jIGluIGRvY3VtZW50czoKICAgICAgICBkb2NfbG93ZXIgPSBkb2MubG93ZXIoKQogICAgICAgIHNjb3JlID0gc3VtKDEgZm9yIHdvcmQgaW4gcXVlc3Rpb25fd29yZHMgaWYgd29yZCBpbiBkb2NfbG93ZXIpCiAgICAgICAgc2NvcmVzLmFwcGVuZChmbG9hdChzY29yZSkpCiAgICAKICAgIHJldHVybiB0b3JjaC50ZW5zb3Ioc2NvcmVzKQoKCmNsYXNzIEJhc2VSZXRyaWV2ZXI6CiAgICAiIiJCYXNlIGNsYXNzIGZvciByZXRyaWV2YWwgc3RyYXRlZ2llcy4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yKToKICAgICAgICBzZWxmLmVtYmVkZGVyID0gZW1iZWRkZXIKICAgICAgICBzZWxmLmxsbSA9IGxsbV9nZW5lcmF0b3IKICAgIAogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGRvY3VtZW50czogTGlzdFtzdHJdLCAKICAgICAgICAgICAgICAgICBkb2NfZW1iZWRkaW5nczogdG9yY2guVGVuc29yLCB0b3BrOiBpbnQgPSA1KSAtPiBUdXBsZVtzdHIsIERpY3RdOgogICAgICAgICIiIgogICAgICAgIFJldHJpZXZlIHJlbGV2YW50IGNvbnRleHQgZnJvbSBkb2N1bWVudHMuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcXVlc3Rpb246IFF1ZXJ5IHRleHQKICAgICAgICAgICAgZG9jdW1lbnRzOiBMaXN0IG9mIGRvY3VtZW50IHN0cmluZ3MKICAgICAgICAgICAgZG9jX2VtYmVkZGluZ3M6IFByZS1jb21wdXRlZCBkb2N1bWVudCBlbWJlZGRpbmdzCiAgICAgICAgICAgIHRvcGs6IE51bWJlciBvZiBkb2N1bWVudHMgdG8gcmV0cmlldmUKICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICB0dXBsZTogKGNvbnRleHRfc3RyaW5nLCByZXRyaWV2YWxfZGV0YWlscykKICAgICAgICAiIiIKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yCiAgICAKICAgIGRlZiBhbnN3ZXIoc2VsZiwgcXVlc3Rpb246IHN0ciwgZG9jdW1lbnRzOiBMaXN0W3N0cl0sCiAgICAgICAgICAgICAgIGRvY19lbWJlZGRpbmdzOiB0b3JjaC5UZW5zb3IsIHRvcGs6IGludCA9IDUsCiAgICAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zOiBpbnQgPSAxMDApIC0+IERpY3Q6CiAgICAgICAgIiIiCiAgICAgICAgRnVsbCBwaXBlbGluZTogcmV0cmlldmUgKyBnZW5lcmF0ZSBhbnN3ZXIuCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgZGljdDogeyJhbnN3ZXIiOiBzdHIsICJjb250ZXh0Ijogc3RyLCAicmV0cmlldmFsX2RldGFpbHMiOiBkaWN0fQogICAgICAgICIiIgogICAgICAgIGNvbnRleHQsIGRldGFpbHMgPSBzZWxmLnJldHJpZXZlKHF1ZXN0aW9uLCBkb2N1bWVudHMsIGRvY19lbWJlZGRpbmdzLCB0b3BrKQogICAgICAgIGFuc3dlciwgcHJvbXB0ID0gc2VsZi5sbG0uZ2VuZXJhdGVfcWEocXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zKQogICAgICAgIAogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJhbnN3ZXIiOiBhbnN3ZXIsCiAgICAgICAgICAgICJjb250ZXh0IjogY29udGV4dCwKICAgICAgICAgICAgInByb21wdCI6IHByb21wdCwKICAgICAgICAgICAgInJldHJpZXZhbF9kZXRhaWxzIjogZGV0YWlscywKICAgICAgICAgICAgIm1vZGUiOiBzZWxmLl9fY2xhc3NfXy5fX25hbWVfXwogICAgICAgIH0KCgpjbGFzcyBQdXJlUkFHKEJhc2VSZXRyaWV2ZXIpOgogICAgIiIiCiAgICBQdXJlIFJBRzogQk0yNSArIEVtYmVkZGluZyByZXRyaWV2YWwgKG5vIE1lbW9yeSkuCiAgICAKICAgIFRoaXMgaXMgdGhlIHRyYWRpdGlvbmFsIFJBRyBiYXNlbGluZSB3aXRob3V0IGFueSBUaXRhbiBtZW1vcnkuCiAgICAiIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yLCAKICAgICAgICAgICAgICAgICBibTI1X3dlaWdodDogZmxvYXQgPSAwLjQsIGVtYmVkX3dlaWdodDogZmxvYXQgPSAwLjYpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IpCiAgICAgICAgc2VsZi5ibTI1X3dlaWdodCA9IGJtMjVfd2VpZ2h0CiAgICAgICAgc2VsZi5lbWJlZF93ZWlnaHQgPSBlbWJlZF93ZWlnaHQKICAgIAogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGRvY3VtZW50czogTGlzdFtzdHJdLAogICAgICAgICAgICAgICAgIGRvY19lbWJlZGRpbmdzOiB0b3JjaC5UZW5zb3IsIHRvcGs6IGludCA9IDUpIC0+IFR1cGxlW3N0ciwgRGljdF06CiAgICAgICAgIiIiUmV0cmlldmUgdXNpbmcgQk0yNSArIEVtYmVkZGluZyBvbmx5LiIiIgogICAgICAgIAogICAgICAgICMgRW1iZWQgcXVlc3Rpb24KICAgICAgICBxdWVyeV9lbWIgPSBzZWxmLmVtYmVkZGVyKHF1ZXN0aW9uKQogICAgICAgIHF1ZXJ5X3ZlYyA9IHF1ZXJ5X2VtYi5yZXNoYXBlKC0xLCBzZWxmLmVtYmVkZGVyLnRhcmdldF9kaW0pLm1lYW4oZGltPTApCiAgICAgICAgCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICMgQk0yNSBzY29yZXMKICAgICAgICAgICAgYm0yNV9zY29yZXMgPSBjb21wdXRlX2JtMjVfc2NvcmVzKHF1ZXN0aW9uLCBkb2N1bWVudHMpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEVtYmVkZGluZyBzaW1pbGFyaXR5CiAgICAgICAgICAgIGVtYmVkX3Njb3JlcyA9IEYuY29zaW5lX3NpbWlsYXJpdHkoCiAgICAgICAgICAgICAgICBxdWVyeV92ZWMudW5zcXVlZXplKDApLCBkb2NfZW1iZWRkaW5ncwogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEZ1c2lvbiAobm8gTWVtb3J5ISkKICAgICAgICAgICAgZnVzZWRfc2NvcmVzID0gKAogICAgICAgICAgICAgICAgc2VsZi5ibTI1X3dlaWdodCAqIG5vcm1hbGl6ZV9zY29yZXMoYm0yNV9zY29yZXMpICsKICAgICAgICAgICAgICAgIHNlbGYuZW1iZWRfd2VpZ2h0ICogbm9ybWFsaXplX3Njb3JlcyhlbWJlZF9zY29yZXMpCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgVG9wLUsKICAgICAgICAgICAgdG9wa192YWx1ZXMsIHRvcGtfaW5kaWNlcyA9IGZ1c2VkX3Njb3Jlcy50b3BrKG1pbih0b3BrLCBsZW4oZG9jdW1lbnRzKSkpCiAgICAgICAgCiAgICAgICAgIyBCdWlsZCBjb250ZXh0CiAgICAgICAgY29udGV4dF9saW5lcyA9IFtkb2N1bWVudHNbaWR4XSBmb3IgaWR4IGluIHRvcGtfaW5kaWNlcy50b2xpc3QoKV0KICAgICAgICBjb250ZXh0ID0gIiAiLmpvaW4oY29udGV4dF9saW5lcykKICAgICAgICAKICAgICAgICByZXR1cm4gY29udGV4dCwgewogICAgICAgICAgICAibW9kZSI6ICJQdXJlUkFHIiwKICAgICAgICAgICAgInRvcGtfaW5kaWNlcyI6IHRvcGtfaW5kaWNlcy50b2xpc3QoKSwKICAgICAgICAgICAgImJtMjVfdG9wMSI6IGJtMjVfc2NvcmVzLmFyZ21heCgpLml0ZW0oKSwKICAgICAgICAgICAgImVtYmVkX3RvcDEiOiBlbWJlZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAiZnVzZWRfdG9wMSI6IGZ1c2VkX3Njb3Jlcy5hcmdtYXgoKS5pdGVtKCkKICAgICAgICB9CgoKY2xhc3MgVGl0YW5Pbmx5KEJhc2VSZXRyaWV2ZXIpOgogICAgIiIiCiAgICBUaXRhbi1Pbmx5OiBNZW1vcnktZ3VpZGVkIHJldHJpZXZhbC4KICAgIAogICAgVXNlcyBvbmx5IHRoZSBUaXRhbiBtZW1vcnkgb3V0cHV0IGZvciByZXRyaWV2YWwgc2NvcmluZy4KICAgIE5vIEJNMjUsIG5vIGRpcmVjdCBlbWJlZGRpbmcgc2ltaWxhcml0eS4KICAgICIiIgogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IsIHRpdGFuX3JhZyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXyhlbWJlZGRlciwgbGxtX2dlbmVyYXRvcikKICAgICAgICBzZWxmLnRpdGFuX3JhZyA9IHRpdGFuX3JhZwogICAgCiAgICBkZWYgcmV0cmlldmUoc2VsZiwgcXVlc3Rpb246IHN0ciwgZG9jdW1lbnRzOiBMaXN0W3N0cl0sCiAgICAgICAgICAgICAgICAgZG9jX2VtYmVkZGluZ3M6IHRvcmNoLlRlbnNvciwgdG9wazogaW50ID0gNSkgLT4gVHVwbGVbc3RyLCBEaWN0XToKICAgICAgICAiIiJSZXRyaWV2ZSB1c2luZyBNZW1vcnkgb3V0cHV0IG9ubHkuIiIiCiAgICAgICAgCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltCiAgICAgICAgCiAgICAgICAgIyBFbWJlZCBxdWVzdGlvbgogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pCiAgICAgICAgcXVlcnlfZmxhdCA9IHF1ZXJ5X2VtYi5yZXNoYXBlKC0xLCBkaW0pCiAgICAgICAgCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICMgR2V0IG1lbW9yeSBvdXRwdXQKICAgICAgICAgICAgbWVtb3J5X291dHB1dCA9IHNlbGYudGl0YW5fcmFnLnRpdGFuLmx0bS5mb3J3YXJkX25vX3VwZGF0ZShxdWVyeV9mbGF0KQogICAgICAgICAgICBtZW1vcnlfdmVjID0gbWVtb3J5X291dHB1dC5tZWFuKGRpbT0wKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBNZW1vcnktYmFzZWQgc2ltaWxhcml0eSAob25seSBzaWduYWwhKQogICAgICAgICAgICBtZW1vcnlfc2NvcmVzID0gRi5jb3NpbmVfc2ltaWxhcml0eSgKICAgICAgICAgICAgICAgIG1lbW9yeV92ZWMudW5zcXVlZXplKDApLCBkb2NfZW1iZWRkaW5ncwogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIFRvcC1LCiAgICAgICAgICAgIHRvcGtfdmFsdWVzLCB0b3BrX2luZGljZXMgPSBtZW1vcnlfc2NvcmVzLnRvcGsobWluKHRvcGssIGxlbihkb2N1bWVudHMpKSkKICAgICAgICAKICAgICAgICAjIEJ1aWxkIGNvbnRleHQKICAgICAgICBjb250ZXh0X2xpbmVzID0gW2RvY3VtZW50c1tpZHhdIGZvciBpZHggaW4gdG9wa19pbmRpY2VzLnRvbGlzdCgpXQogICAgICAgIGNvbnRleHQgPSAiICIuam9pbihjb250ZXh0X2xpbmVzKQogICAgICAgIAogICAgICAgIHJldHVybiBjb250ZXh0LCB7CiAgICAgICAgICAgICJtb2RlIjogIlRpdGFuT25seSIsCiAgICAgICAgICAgICJ0b3BrX2luZGljZXMiOiB0b3BrX2luZGljZXMudG9saXN0KCksCiAgICAgICAgICAgICJtZW1vcnlfdG9wMSI6IG1lbW9yeV9zY29yZXMuYXJnbWF4KCkuaXRlbSgpCiAgICAgICAgfQogICAgCiAgICBkZWYgb25saW5lX2xlYXJuKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGV4cGVjdGVkX2Fuc3dlcjogc3RyKToKICAgICAgICAiIiJMZWFybiBmcm9tIFEmQSBwYWlyIChPbmxpbmUgTGVhcm5pbmcpLiIiIgogICAgICAgIGRpbSA9IHNlbGYuZW1iZWRkZXIudGFyZ2V0X2RpbQogICAgICAgIAogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pCiAgICAgICAgcXVlcnlfdmVjID0gcXVlcnlfZW1iLnJlc2hhcGUoLTEsIGRpbSkubWVhbihkaW09MCwga2VlcGRpbT1UcnVlKQogICAgICAgIAogICAgICAgIGFuc3dlcl9lbWIgPSBzZWxmLmVtYmVkZGVyKGV4cGVjdGVkX2Fuc3dlcikKICAgICAgICBhbnN3ZXJfdmVjID0gYW5zd2VyX2VtYi5yZXNoYXBlKC0xLCBkaW0pLm1lYW4oZGltPTAsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAKICAgICAgICBzZWxmLnRpdGFuX3JhZy50aXRhbi5sdG0uZm9yd2FyZF93aXRoX3VwZGF0ZShxdWVyeV92ZWMsIGFuc3dlcl92ZWMpCgoKY2xhc3MgSHlicmlkUkFHKEJhc2VSZXRyaWV2ZXIpOgogICAgIiIiCiAgICBIeWJyaWQgUkFHOiBCTTI1ICsgTWVtb3J5ICsgRW1iZWRkaW5nIGZ1c2lvbiB3aXRoIGR5bmFtaWMgd2VpZ2h0aW5nLgogICAgCiAgICBLZXkgaW1wcm92ZW1lbnQ6IE1lbW9yeSB3ZWlnaHQgaXMgYWRqdXN0ZWQgZHluYW1pY2FsbHkgYmFzZWQgb24gCiAgICBtZW1vcnkgY29uZmlkZW5jZSAob3V0cHV0IHN0cmVuZ3RoKS4gTG93IGNvbmZpZGVuY2UgPSByZWR1Y2UgbWVtb3J5IHdlaWdodC4KICAgICIiIgogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IsIHRpdGFuX3JhZywKICAgICAgICAgICAgICAgICBiYXNlX2JtMjVfd2VpZ2h0OiBmbG9hdCA9IDAuMywKICAgICAgICAgICAgICAgICBiYXNlX21lbW9yeV93ZWlnaHQ6IGZsb2F0ID0gMC4zLAogICAgICAgICAgICAgICAgIGJhc2VfZW1iZWRfd2VpZ2h0OiBmbG9hdCA9IDAuNCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXyhlbWJlZGRlciwgbGxtX2dlbmVyYXRvcikKICAgICAgICBzZWxmLnRpdGFuX3JhZyA9IHRpdGFuX3JhZwogICAgICAgIHNlbGYuYmFzZV9ibTI1X3dlaWdodCA9IGJhc2VfYm0yNV93ZWlnaHQKICAgICAgICBzZWxmLmJhc2VfbWVtb3J5X3dlaWdodCA9IGJhc2VfbWVtb3J5X3dlaWdodAogICAgICAgIHNlbGYuYmFzZV9lbWJlZF93ZWlnaHQgPSBiYXNlX2VtYmVkX3dlaWdodAogICAgCiAgICBkZWYgX2NvbXB1dGVfbWVtb3J5X2NvbmZpZGVuY2Uoc2VsZiwgcXVlcnlfZW1iLCBtZW1vcnlfb3V0cHV0KToKICAgICAgICAiIiIKICAgICAgICBDb21wdXRlIG1lbW9yeSBjb25maWRlbmNlIGJhc2VkIG9uIG91dHB1dCBzdHJlbmd0aC4KICAgICAgICAKICAgICAgICBIaWdoZXIgY29uZmlkZW5jZSB3aGVuIG1lbW9yeSBvdXRwdXQgbm9ybSBpcyBzaW1pbGFyIHRvIGlucHV0IG5vcm0uCiAgICAgICAgUmV0dXJucyB2YWx1ZSBpbiBbMCwgMV0gd2hlcmUgMSA9IGhpZ2ggY29uZmlkZW5jZS4KICAgICAgICAiIiIKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgaW5wdXRfbm9ybSA9IHRvcmNoLm5vcm0ocXVlcnlfZW1iKQogICAgICAgICAgICBvdXRwdXRfbm9ybSA9IHRvcmNoLm5vcm0obWVtb3J5X291dHB1dCkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgUmF0aW8gb2Ygb3V0cHV0IHRvIGlucHV0IG5vcm0KICAgICAgICAgICAgY29uZmlkZW5jZV9yYXRpbyA9IG91dHB1dF9ub3JtIC8gKGlucHV0X25vcm0gKyAxZS04KQogICAgICAgICAgICAKICAgICAgICAgICAgIyBOb3JtYWxpemUgdG8gMC0xIHJhbmdlIChhc3N1bWluZyB0eXBpY2FsIHJhdGlvIGlzIDAtMikKICAgICAgICAgICAgY29uZmlkZW5jZSA9IHRvcmNoLmNsYW1wKGNvbmZpZGVuY2VfcmF0aW8gLyAyLjAsIDAuMCwgMS4wKS5pdGVtKCkKICAgICAgICAgICAgCiAgICAgICAgICAgIHJldHVybiBjb25maWRlbmNlCiAgICAKICAgIGRlZiByZXRyaWV2ZShzZWxmLCBxdWVzdGlvbjogc3RyLCBkb2N1bWVudHM6IExpc3Rbc3RyXSwKICAgICAgICAgICAgICAgICBkb2NfZW1iZWRkaW5nczogdG9yY2guVGVuc29yLCB0b3BrOiBpbnQgPSA1KSAtPiBUdXBsZVtzdHIsIERpY3RdOgogICAgICAgICIiIlJldHJpZXZlIHVzaW5nIEJNMjUgKyBNZW1vcnkgKyBFbWJlZGRpbmcgZnVzaW9uIHdpdGggZHluYW1pYyB3ZWlnaHRzLiIiIgogICAgICAgIAogICAgICAgIGRpbSA9IHNlbGYuZW1iZWRkZXIudGFyZ2V0X2RpbQogICAgICAgIAogICAgICAgICMgRW1iZWQgcXVlc3Rpb24KICAgICAgICBxdWVyeV9lbWIgPSBzZWxmLmVtYmVkZGVyKHF1ZXN0aW9uKQogICAgICAgIHF1ZXJ5X2ZsYXQgPSBxdWVyeV9lbWIucmVzaGFwZSgtMSwgZGltKQogICAgICAgIHF1ZXJ5X3ZlYyA9IHF1ZXJ5X2ZsYXQubWVhbihkaW09MCkKICAgICAgICAKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgIyBTaWduYWwgMTogQk0yNQogICAgICAgICAgICBibTI1X3Njb3JlcyA9IGNvbXB1dGVfYm0yNV9zY29yZXMocXVlc3Rpb24sIGRvY3VtZW50cykKICAgICAgICAgICAgCiAgICAgICAgICAgICMgU2lnbmFsIDI6IE1lbW9yeQogICAgICAgICAgICBtZW1vcnlfb3V0cHV0ID0gc2VsZi50aXRhbl9yYWcudGl0YW4ubHRtLmZvcndhcmRfbm9fdXBkYXRlKHF1ZXJ5X2ZsYXQpCiAgICAgICAgICAgIG1lbW9yeV92ZWMgPSBtZW1vcnlfb3V0cHV0Lm1lYW4oZGltPTApCiAgICAgICAgICAgIG1lbW9yeV9zY29yZXMgPSBGLmNvc2luZV9zaW1pbGFyaXR5KAogICAgICAgICAgICAgICAgbWVtb3J5X3ZlYy51bnNxdWVlemUoMCksIGRvY19lbWJlZGRpbmdzCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgU2lnbmFsIDM6IERpcmVjdCBFbWJlZGRpbmcKICAgICAgICAgICAgZW1iZWRfc2NvcmVzID0gRi5jb3NpbmVfc2ltaWxhcml0eSgKICAgICAgICAgICAgICAgIHF1ZXJ5X3ZlYy51bnNxdWVlemUoMCksIGRvY19lbWJlZGRpbmdzCiAgICAgICAgICAgICkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgQ29tcHV0ZSBtZW1vcnkgY29uZmlkZW5jZQogICAgICAgICAgICBtZW1vcnlfY29uZmlkZW5jZSA9IHNlbGYuX2NvbXB1dGVfbWVtb3J5X2NvbmZpZGVuY2UocXVlcnlfZmxhdCwgbWVtb3J5X291dHB1dCkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgRHluYW1pYyB3ZWlnaHQgYWRqdXN0bWVudCBiYXNlZCBvbiBjb25maWRlbmNlCiAgICAgICAgICAgICMgTG93IGNvbmZpZGVuY2Ug4oaSIHJlZHVjZSBtZW1vcnkgd2VpZ2h0LCBpbmNyZWFzZSBlbWJlZGRpbmcgd2VpZ2h0CiAgICAgICAgICAgIG1lbW9yeV93ZWlnaHQgPSBzZWxmLmJhc2VfbWVtb3J5X3dlaWdodCAqIG1lbW9yeV9jb25maWRlbmNlCiAgICAgICAgICAgIGVtYmVkX3dlaWdodCA9IHNlbGYuYmFzZV9lbWJlZF93ZWlnaHQgKyBzZWxmLmJhc2VfbWVtb3J5X3dlaWdodCAqICgxIC0gbWVtb3J5X2NvbmZpZGVuY2UpCiAgICAgICAgICAgIGJtMjVfd2VpZ2h0ID0gc2VsZi5iYXNlX2JtMjVfd2VpZ2h0CiAgICAgICAgICAgIAogICAgICAgICAgICAjIE5vcm1hbGl6ZSB3ZWlnaHRzIHRvIHN1bSB0byAxCiAgICAgICAgICAgIHRvdGFsID0gYm0yNV93ZWlnaHQgKyBtZW1vcnlfd2VpZ2h0ICsgZW1iZWRfd2VpZ2h0CiAgICAgICAgICAgIGJtMjVfd2VpZ2h0IC89IHRvdGFsCiAgICAgICAgICAgIG1lbW9yeV93ZWlnaHQgLz0gdG90YWwKICAgICAgICAgICAgZW1iZWRfd2VpZ2h0IC89IHRvdGFsCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEZ1c2lvbiB3aXRoIGR5bmFtaWMgd2VpZ2h0cwogICAgICAgICAgICBmdXNlZF9zY29yZXMgPSAoCiAgICAgICAgICAgICAgICBibTI1X3dlaWdodCAqIG5vcm1hbGl6ZV9zY29yZXMoYm0yNV9zY29yZXMpICsKICAgICAgICAgICAgICAgIG1lbW9yeV93ZWlnaHQgKiBub3JtYWxpemVfc2NvcmVzKG1lbW9yeV9zY29yZXMpICsKICAgICAgICAgICAgICAgIGVtYmVkX3dlaWdodCAqIG5vcm1hbGl6ZV9zY29yZXMoZW1iZWRfc2NvcmVzKQogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIFRvcC1LCiAgICAgICAgICAgIHRvcGtfdmFsdWVzLCB0b3BrX2luZGljZXMgPSBmdXNlZF9zY29yZXMudG9wayhtaW4odG9waywgbGVuKGRvY3VtZW50cykpKQogICAgICAgIAogICAgICAgICMgQnVpbGQgY29udGV4dAogICAgICAgIGNvbnRleHRfbGluZXMgPSBbZG9jdW1lbnRzW2lkeF0gZm9yIGlkeCBpbiB0b3BrX2luZGljZXMudG9saXN0KCldCiAgICAgICAgY29udGV4dCA9ICIgIi5qb2luKGNvbnRleHRfbGluZXMpCiAgICAgICAgCiAgICAgICAgcmV0dXJuIGNvbnRleHQsIHsKICAgICAgICAgICAgIm1vZGUiOiAiSHlicmlkUkFHIiwKICAgICAgICAgICAgInRvcGtfaW5kaWNlcyI6IHRvcGtfaW5kaWNlcy50b2xpc3QoKSwKICAgICAgICAgICAgImJtMjVfdG9wMSI6IGJtMjVfc2NvcmVzLmFyZ21heCgpLml0ZW0oKSwKICAgICAgICAgICAgIm1lbW9yeV90b3AxIjogbWVtb3J5X3Njb3Jlcy5hcmdtYXgoKS5pdGVtKCksCiAgICAgICAgICAgICJlbWJlZF90b3AxIjogZW1iZWRfc2NvcmVzLmFyZ21heCgpLml0ZW0oKSwKICAgICAgICAgICAgImZ1c2VkX3RvcDEiOiBmdXNlZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAibWVtb3J5X2NvbmZpZGVuY2UiOiBtZW1vcnlfY29uZmlkZW5jZSwKICAgICAgICAgICAgImR5bmFtaWNfd2VpZ2h0cyI6IHsKICAgICAgICAgICAgICAgICJibTI1Ijogcm91bmQoYm0yNV93ZWlnaHQsIDMpLAogICAgICAgICAgICAgICAgIm1lbW9yeSI6IHJvdW5kKG1lbW9yeV93ZWlnaHQsIDMpLAogICAgICAgICAgICAgICAgImVtYmVkIjogcm91bmQoZW1iZWRfd2VpZ2h0LCAzKQogICAgICAgICAgICB9CiAgICAgICAgfQogICAgCiAgICBkZWYgb25saW5lX2xlYXJuKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGV4cGVjdGVkX2Fuc3dlcjogc3RyKToKICAgICAgICAiIiJMZWFybiBmcm9tIFEmQSBwYWlyIChPbmxpbmUgTGVhcm5pbmcpLiIiIgogICAgICAgIGRpbSA9IHNlbGYuZW1iZWRkZXIudGFyZ2V0X2RpbQogICAgICAgIAogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pCiAgICAgICAgcXVlcnlfdmVjID0gcXVlcnlfZW1iLnJlc2hhcGUoLTEsIGRpbSkubWVhbihkaW09MCwga2VlcGRpbT1UcnVlKQogICAgICAgIAogICAgICAgIGFuc3dlcl9lbWIgPSBzZWxmLmVtYmVkZGVyKGV4cGVjdGVkX2Fuc3dlcikKICAgICAgICBhbnN3ZXJfdmVjID0gYW5zd2VyX2VtYi5yZXNoYXBlKC0xLCBkaW0pLm1lYW4oZGltPTAsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAKICAgICAgICBzZWxmLnRpdGFuX3JhZy50aXRhbi5sdG0uZm9yd2FyZF93aXRoX3VwZGF0ZShxdWVyeV92ZWMsIGFuc3dlcl92ZWMpCgoKZGVmIGNyZWF0ZV9yZXRyaWV2ZXJzKGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yLCB0aXRhbl9yYWc9Tm9uZSkgLT4gRGljdFtzdHIsIEJhc2VSZXRyaWV2ZXJdOgogICAgIiIiCiAgICBGYWN0b3J5IGZ1bmN0aW9uIHRvIGNyZWF0ZSBhbGwgcmV0cmlldmVyIGluc3RhbmNlcy4KICAgIAogICAgQXJnczoKICAgICAgICBlbWJlZGRlcjogU2VudGVuY2VUcmFuc2Zvcm1lckVtYmVkZGVyIGluc3RhbmNlCiAgICAgICAgbGxtX2dlbmVyYXRvcjogRmxhblQ1R2VuZXJhdG9yIGluc3RhbmNlCiAgICAgICAgdGl0YW5fcmFnOiBUaXRhblJBRyBpbnN0YW5jZSAocmVxdWlyZWQgZm9yIFRpdGFuT25seSBhbmQgSHlicmlkUkFHKQogICAgCiAgICBSZXR1cm5zOgogICAgICAgIGRpY3Q6IHsicHVyZV9yYWciOiBQdXJlUkFHLCAidGl0YW5fb25seSI6IFRpdGFuT25seSwgImh5YnJpZCI6IEh5YnJpZFJBR30KICAgICIiIgogICAgcmV0cmlldmVycyA9IHsKICAgICAgICAicHVyZV9yYWciOiBQdXJlUkFHKGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yKQogICAgfQogICAgCiAgICBpZiB0aXRhbl9yYWcgaXMgbm90IE5vbmU6CiAgICAgICAgcmV0cmlldmVyc1sidGl0YW5fb25seSJdID0gVGl0YW5Pbmx5KGVtYmVkZGVyLCBsbG1fZ2VuZXJhdG9yLCB0aXRhbl9yYWcpCiAgICAgICAgcmV0cmlldmVyc1siaHlicmlkIl0gPSBIeWJyaWRSQUcoZW1iZWRkZXIsIGxsbV9nZW5lcmF0b3IsIHRpdGFuX3JhZykKICAgIAogICAgcmV0dXJuIHJldHJpZXZlcnMK',
    'projects/hybrid_titans/compare_baselines.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKQmFzZWxpbmUgQ29tcGFyaXNvbiBTY3JpcHQgZm9yIFRpdGFuUkFHLgoKQ29tcGFyZXMgdGhyZWUgcmV0cmlldmFsIHN0cmF0ZWdpZXM6Ci0gUHVyZVJBRzogQk0yNSArIEVtYmVkZGluZyAobm8gTWVtb3J5KQotIFRpdGFuT25seTogTWVtb3J5LWd1aWRlZCByZXRyaWV2YWwgb25seSAgCi0gSHlicmlkUkFHOiBCTTI1ICsgTWVtb3J5ICsgRW1iZWRkaW5nIGZ1c2lvbgoKVXNhZ2U6CiAgICBweXRob24gY29tcGFyZV9iYXNlbGluZXMucHkgLS1lc3NheSBjbGltYXRlIC0tZXBvY2hzIDUwCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzeXMKaW1wb3J0IG9zCmltcG9ydCByZQoKIyBBZGQgcGF0aHMKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSwgJ3NyYycpKQoKZnJvbSBjb21tb24uZW1iZWRkZXJzIGltcG9ydCBTZW50ZW5jZVRyYW5zZm9ybWVyRW1iZWRkZXIKZnJvbSBjb21tb24udGl0YW5fdXRpbHMgaW1wb3J0IGNyZWF0ZV90aXRhbl9yYWcsIGRpZ2VzdF9kb2N1bWVudApmcm9tIGNvbW1vbi5sbG1fdXRpbHMgaW1wb3J0IEZsYW5UNUdlbmVyYXRvcgpmcm9tIGNvbW1vbi50ZXh0X3V0aWxzIGltcG9ydCBzcGxpdF9pbnRvX2NodW5rcwpmcm9tIGJhc2VsaW5lcyBpbXBvcnQgY3JlYXRlX3JldHJpZXZlcnMKCgojIEltcG9ydCBlc3NheXMgZnJvbSBkYXRhIGRpcmVjdG9yeQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSksICdkYXRhJykpCmZyb20gc2FtcGxlX2Vzc2F5cyBpbXBvcnQgZ2V0X2FsbF9lc3NheXMsIGdldF90ZXN0X3F1ZXN0aW9ucwoKRVNTQVlTID0gZ2V0X2FsbF9lc3NheXMoKQpURVNUX1FVRVNUSU9OUyA9IGdldF90ZXN0X3F1ZXN0aW9ucygpCgoKZGVmIGV4dHJhY3Rfa2V5X2VsZW1lbnRzKHRleHQpOgogICAgIiIiRXh0cmFjdCBrZXkgbnVtYmVycyBhbmQga2V5d29yZHMgZnJvbSB0ZXh0IGZvciBtYXRjaGluZy4iIiIKICAgIHRleHQgPSB0ZXh0Lmxvd2VyKCkKICAgICMgRXh0cmFjdCBudW1iZXJzIChpbmNsdWRpbmcgZGVjaW1hbHMgYW5kIHBlcmNlbnRhZ2VzKQogICAgbnVtYmVycyA9IHJlLmZpbmRhbGwocidbXGQuXSslPycsIHRleHQpCiAgICAjIEV4dHJhY3QgeWVhcnMgKDQtZGlnaXQgbnVtYmVycykKICAgIHllYXJzID0gcmUuZmluZGFsbChyJ1xiKDE5fDIwKVxkezJ9XGInLCB0ZXh0KQogICAgcmV0dXJuIHNldChudW1iZXJzICsgeWVhcnMpCgoKZGVmIGV4dHJhY3Rfa2V5d29yZHModGV4dCk6CiAgICAiIiJFeHRyYWN0IG1lYW5pbmdmdWwga2V5d29yZHMgZnJvbSB0ZXh0LCByZW1vdmluZyBzdG9wd29yZHMuIiIiCiAgICBzdG9wd29yZHMgPSB7J2EnLCAnYW4nLCAndGhlJywgJ2lzJywgJ2FyZScsICd3YXMnLCAnd2VyZScsICdiZScsICdiZWVuJywgJ2JlaW5nJywKICAgICAgICAgICAgICAgICAnaGF2ZScsICdoYXMnLCAnaGFkJywgJ2RvJywgJ2RvZXMnLCAnZGlkJywgJ3dpbGwnLCAnd291bGQnLCAnY291bGQnLAogICAgICAgICAgICAgICAgICdzaG91bGQnLCAnbWF5JywgJ21pZ2h0JywgJ211c3QnLCAnc2hhbGwnLCAnY2FuJywgJ25lZWQnLCAnZGFyZScsCiAgICAgICAgICAgICAgICAgJ3RvJywgJ29mJywgJ2luJywgJ2ZvcicsICdvbicsICd3aXRoJywgJ2F0JywgJ2J5JywgJ2Zyb20nLCAnYXMnLAogICAgICAgICAgICAgICAgICdpbnRvJywgJ3Rocm91Z2gnLCAnZHVyaW5nJywgJ2JlZm9yZScsICdhZnRlcicsICdhYm92ZScsICdiZWxvdycsCiAgICAgICAgICAgICAgICAgJ2JldHdlZW4nLCAndW5kZXInLCAnYWdhaW4nLCAnZnVydGhlcicsICd0aGVuJywgJ29uY2UnLCAnYW5kJywgJ29yJywKICAgICAgICAgICAgICAgICAnYnV0JywgJ2lmJywgJ2JlY2F1c2UnLCAndW50aWwnLCAnd2hpbGUnLCAndGhhdCcsICd3aGljaCcsICd3aG8nLAogICAgICAgICAgICAgICAgICd3aG9tJywgJ3RoaXMnLCAndGhlc2UnLCAndGhvc2UnLCAnaXQnLCAnaXRzJywgJ3RoZWlyJywgJ3RoZXknfQogICAgCiAgICAjIFJlbW92ZSBwdW5jdHVhdGlvbiBhbmQgc3BsaXQKICAgIHRleHQgPSByZS5zdWIocidbXlx3XHNdJywgJyAnLCB0ZXh0Lmxvd2VyKCkpCiAgICB3b3JkcyA9IHRleHQuc3BsaXQoKQogICAgCiAgICAjIEZpbHRlciBzdG9wd29yZHMgYW5kIHNob3J0IHdvcmRzCiAgICBrZXl3b3JkcyA9IFt3IGZvciB3IGluIHdvcmRzIGlmIHcgbm90IGluIHN0b3B3b3JkcyBhbmQgbGVuKHcpID4gMl0KICAgIHJldHVybiBzZXQoa2V5d29yZHMpCgoKZGVmIGV2YWx1YXRlX2Fuc3dlcl9xdWFsaXR5KGV4cGVjdGVkLCBnb3QpOgogICAgIiIiCiAgICBGbGV4aWJsZSBhbnN3ZXIgZXZhbHVhdGlvbiB1c2luZyBtdWx0aXBsZSBjcml0ZXJpYS4KICAgIAogICAgUmV0dXJucyBUcnVlIGlmIGFueSBvZiB0aGVzZSBjb25kaXRpb25zIGFyZSBtZXQ6CiAgICAxLiBCaWRpcmVjdGlvbmFsIGNvbnRhaW5tZW50CiAgICAyLiBQaHJhc2Ugb3ZlcmxhcCAoa2V5IHBocmFzZXMgZnJvbSBnb3QgYXJlIGluIGV4cGVjdGVkKQogICAgMy4gS2V5IG51bWJlcnMveWVhcnMgbWF0Y2gKICAgIDQuIEtleXdvcmQgb3ZlcmxhcCA+PSA0MCUKICAgICIiIgogICAgZXhwZWN0ZWRfbG93ZXIgPSBleHBlY3RlZC5sb3dlcigpCiAgICBnb3RfbG93ZXIgPSBnb3QubG93ZXIoKQogICAgCiAgICAjIDEuIEJpZGlyZWN0aW9uYWwgY29udGFpbm1lbnQKICAgIGlmIGV4cGVjdGVkX2xvd2VyIGluIGdvdF9sb3dlciBvciBnb3RfbG93ZXIgaW4gZXhwZWN0ZWRfbG93ZXI6CiAgICAgICAgcmV0dXJuIFRydWUKICAgIAogICAgIyAyLiBQaHJhc2Ugb3ZlcmxhcCAtIGNoZWNrIGlmIDMrIGNvbnNlY3V0aXZlIHdvcmRzIGZyb20gZ290IGFwcGVhciBpbiBleHBlY3RlZAogICAgZ290X3dvcmRzID0gZ290X2xvd2VyLnNwbGl0KCkKICAgIGlmIGxlbihnb3Rfd29yZHMpID49IDM6CiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKGdvdF93b3JkcykgLSAyKToKICAgICAgICAgICAgcGhyYXNlID0gJyAnLmpvaW4oZ290X3dvcmRzW2k6aSszXSkKICAgICAgICAgICAgaWYgcGhyYXNlIGluIGV4cGVjdGVkX2xvd2VyOgogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgIAogICAgIyAzLiBLZXkgZWxlbWVudHMgbWF0Y2hpbmcgKG51bWJlcnMsIHllYXJzKQogICAgZXhwZWN0ZWRfa2V5cyA9IGV4dHJhY3Rfa2V5X2VsZW1lbnRzKGV4cGVjdGVkKQogICAgZ290X2tleXMgPSBleHRyYWN0X2tleV9lbGVtZW50cyhnb3QpCiAgICAKICAgIGlmIGV4cGVjdGVkX2tleXMgYW5kIGV4cGVjdGVkX2tleXMgJiBnb3Rfa2V5czoKICAgICAgICByZXR1cm4gVHJ1ZQogICAgCiAgICAjIDQuIEtleXdvcmQgb3ZlcmxhcCBtYXRjaGluZyAobG93ZXJlZCB0byA0MCUpCiAgICBleHBlY3RlZF9rZXl3b3JkcyA9IGV4dHJhY3Rfa2V5d29yZHMoZXhwZWN0ZWQpCiAgICBnb3Rfa2V5d29yZHMgPSBleHRyYWN0X2tleXdvcmRzKGdvdCkKICAgIAogICAgaWYgZXhwZWN0ZWRfa2V5d29yZHM6CiAgICAgICAgb3ZlcmxhcCA9IGxlbihleHBlY3RlZF9rZXl3b3JkcyAmIGdvdF9rZXl3b3JkcykKICAgICAgICBvdmVybGFwX3JhdGlvID0gb3ZlcmxhcCAvIGxlbihleHBlY3RlZF9rZXl3b3JkcykKICAgICAgICAKICAgICAgICBpZiBvdmVybGFwX3JhdGlvID49IDAuNDoKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgIAogICAgcmV0dXJuIEZhbHNlCgoKZGVmIGV2YWx1YXRlX3JldHJpZXZlcihyZXRyaWV2ZXIsIGRvY3VtZW50cywgZG9jX2VtYmVkZGluZ3MsIHF1ZXN0aW9ucywgdmVyYm9zZT1UcnVlKToKICAgICIiIkV2YWx1YXRlIGEgcmV0cmlldmVyIG9uIGEgc2V0IG9mIHF1ZXN0aW9ucy4iIiIKICAgIGNvcnJlY3QgPSAwCiAgICB0b3RhbCA9IGxlbihxdWVzdGlvbnMpCiAgICAKICAgIHJlc3VsdHMgPSBbXQogICAgZm9yIHF1ZXN0aW9uLCBleHBlY3RlZCBpbiBxdWVzdGlvbnM6CiAgICAgICAgcmVzdWx0ID0gcmV0cmlldmVyLmFuc3dlcihxdWVzdGlvbiwgZG9jdW1lbnRzLCBkb2NfZW1iZWRkaW5ncywgdG9waz0zKQogICAgICAgIAogICAgICAgICMgRmxleGlibGUgZXZhbHVhdGlvbgogICAgICAgIGlzX2NvcnJlY3QgPSBldmFsdWF0ZV9hbnN3ZXJfcXVhbGl0eShleHBlY3RlZCwgcmVzdWx0WyJhbnN3ZXIiXSkKICAgICAgICBpZiBpc19jb3JyZWN0OgogICAgICAgICAgICBjb3JyZWN0ICs9IDEKICAgICAgICAKICAgICAgICByZXN1bHRzLmFwcGVuZCh7CiAgICAgICAgICAgICJxdWVzdGlvbiI6IHF1ZXN0aW9uLAogICAgICAgICAgICAiZXhwZWN0ZWQiOiBleHBlY3RlZCwKICAgICAgICAgICAgImFuc3dlciI6IHJlc3VsdFsiYW5zd2VyIl0sCiAgICAgICAgICAgICJjb3JyZWN0IjogaXNfY29ycmVjdCwKICAgICAgICAgICAgIm1vZGUiOiByZXN1bHRbIm1vZGUiXQogICAgICAgIH0pCiAgICAgICAgCiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgc3RhdHVzID0gIuKchSIgaWYgaXNfY29ycmVjdCBlbHNlICLinYwiCiAgICAgICAgICAgIHByaW50KGYiICB7c3RhdHVzfSBROiB7cXVlc3Rpb259IikKICAgICAgICAgICAgcHJpbnQoZiIgICAgIEV4cGVjdGVkOiB7ZXhwZWN0ZWR9IikKICAgICAgICAgICAgcHJpbnQoZiIgICAgIEdvdDoge3Jlc3VsdFsnYW5zd2VyJ119IikKICAgICAgICAgICAgIyBTaG93IHJldHJpZXZhbCBkZXRhaWxzIGlmIGF2YWlsYWJsZQogICAgICAgICAgICBkZXRhaWxzID0gcmVzdWx0LmdldCgicmV0cmlldmFsX2RldGFpbHMiLCB7fSkKICAgICAgICAgICAgaWYgIm1lbW9yeV9jb25maWRlbmNlIiBpbiBkZXRhaWxzOgogICAgICAgICAgICAgICAgY29uZiA9IGRldGFpbHNbIm1lbW9yeV9jb25maWRlbmNlIl0KICAgICAgICAgICAgICAgIHdlaWdodHMgPSBkZXRhaWxzLmdldCgiZHluYW1pY193ZWlnaHRzIiwge30pCiAgICAgICAgICAgICAgICBwcmludChmIiAgICAgW0NvbmZpZGVuY2U6IHtjb25mOi4yZn0gfCBXZWlnaHRzOiBibTI1PXt3ZWlnaHRzLmdldCgnYm0yNScsIDApOi4yZn0sIG1lbT17d2VpZ2h0cy5nZXQoJ21lbW9yeScsIDApOi4yZn0sIGVtYj17d2VpZ2h0cy5nZXQoJ2VtYmVkJywgMCk6LjJmfV0iKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBTaG93IHJldHJpZXZlZCBjaHVua3MgZm9yIHdyb25nIGFuc3dlcnMKICAgICAgICAgICAgaWYgbm90IGlzX2NvcnJlY3QgYW5kICJjb250ZXh0IiBpbiByZXN1bHQ6CiAgICAgICAgICAgICAgICBwcmludCgiICAgICDwn5OEIFJldHJpZXZlZCBjb250ZXh0OiIpCiAgICAgICAgICAgICAgICBjb250ZXh0ID0gcmVzdWx0WyJjb250ZXh0Il0KICAgICAgICAgICAgICAgICMgU2hvdyBmaXJzdCAzMDAgY2hhcnMKICAgICAgICAgICAgICAgIHByaW50KGYiICAgICAgICB7Y29udGV4dFs6MzAwXX0uLi4iKQogICAgCiAgICBhY2N1cmFjeSA9IGNvcnJlY3QgLyB0b3RhbCAqIDEwMAogICAgcmV0dXJuIGFjY3VyYWN5LCByZXN1bHRzCgoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJDb21wYXJlIGJhc2VsaW5lIHJldHJpZXZhbCBzdHJhdGVnaWVzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXNzYXkiLCB0eXBlPXN0ciwgZGVmYXVsdD0iY2xpbWF0ZSIsIAogICAgICAgICAgICAgICAgICAgICAgIGNob2ljZXM9bGlzdChFU1NBWVMua2V5cygpKSwgaGVscD0iRXNzYXkgdG8gdXNlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NTAsIGhlbHA9IkRpZ2VzdGlvbiBlcG9jaHMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10b3BrIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NSwgaGVscD0iVG9wLUsgZm9yIHJldHJpZXZhbCIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgCiAgICBwcmludCgiPSIgKiA2MCkKICAgIHByaW50KCJCQVNFTElORSBDT01QQVJJU09OOiBQdXJlUkFHIHZzIFRpdGFuT25seSB2cyBIeWJyaWRSQUciKQogICAgcHJpbnQoIj0iICogNjApCiAgICAKICAgICMgSW5pdGlhbGl6ZSBjb21wb25lbnRzCiAgICBwcmludCgiXG7wn5OmIExvYWRpbmcgY29tcG9uZW50cy4uLiIpCiAgICBlbWJlZGRlciA9IFNlbnRlbmNlVHJhbnNmb3JtZXJFbWJlZGRlcih0YXJnZXRfZGltPTI1NikKICAgIGxsbSA9IEZsYW5UNUdlbmVyYXRvcigiZ29vZ2xlL2ZsYW4tdDUtbGFyZ2UiKQogICAgdGl0YW5fcmFnID0gY3JlYXRlX3RpdGFuX3JhZyhkaW09MjU2KQogICAgCiAgICAjIExvYWQgZXNzYXkKICAgIHRleHQgPSBFU1NBWVNbYXJncy5lc3NheV0KICAgIHF1ZXN0aW9ucyA9IFRFU1RfUVVFU1RJT05TW2FyZ3MuZXNzYXldCiAgICAKICAgICMgUGFyc2UgaW50byBzZW1hbnRpYyBjaHVua3MgKGluc3RlYWQgb2Ygc2ltcGxlIGxpbmUgc3BsaXQpCiAgICBjaHVua3MgPSBzcGxpdF9pbnRvX2NodW5rcyh0ZXh0LCBzZW50ZW5jZXNfcGVyX2NodW5rPTMsIG92ZXJsYXBfc2VudGVuY2VzPTEpCiAgICBwcmludChmIiAgIFNwbGl0IGludG8ge2xlbihjaHVua3MpfSBzZW1hbnRpYyBjaHVua3MiKQogICAgCiAgICAjIERpZ2VzdCBkb2N1bWVudAogICAgcHJpbnQoZiJcbvCfp6AgRGlnZXN0aW5nIGRvY3VtZW50ICh7YXJncy5lcG9jaHN9IGVwb2NocykuLi4iKQogICAgY2h1bmtfZW1iZWRkaW5ncyA9IGVtYmVkZGVyLmVtYmVkX2JhdGNoKGNodW5rcykKICAgIAogICAgZm9yIGVwb2NoIGluIHJhbmdlKGFyZ3MuZXBvY2hzKToKICAgICAgICBhbGxfZW1iID0gZW1iZWRkZXIuZW1iZWRfYmF0Y2goY2h1bmtzKQogICAgICAgIHRpdGFuX3JhZy5kaWdlc3Rfa25vd2xlZGdlKGFsbF9lbWIudW5zcXVlZXplKDApKQogICAgICAgIGlmIChlcG9jaCArIDEpICUgMTAgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiIgICBFcG9jaCB7ZXBvY2ggKyAxfS97YXJncy5lcG9jaHN9IikKICAgIAogICAgIyBDcmVhdGUgcmV0cmlldmVycwogICAgcmV0cmlldmVycyA9IGNyZWF0ZV9yZXRyaWV2ZXJzKGVtYmVkZGVyLCBsbG0sIHRpdGFuX3JhZykKICAgIAogICAgIyBFdmFsdWF0ZSBlYWNoCiAgICBwcmludCgiXG4iICsgIj0iICogNjApCiAgICByZXN1bHRzX3N1bW1hcnkgPSB7fQogICAgCiAgICBmb3IgbmFtZSwgcmV0cmlldmVyIGluIHJldHJpZXZlcnMuaXRlbXMoKToKICAgICAgICBwcmludChmIlxu8J+TiiBFdmFsdWF0aW5nOiB7bmFtZS51cHBlcigpfSIpCiAgICAgICAgcHJpbnQoIi0iICogNDApCiAgICAgICAgYWNjdXJhY3ksIHJlc3VsdHMgPSBldmFsdWF0ZV9yZXRyaWV2ZXIoCiAgICAgICAgICAgIHJldHJpZXZlciwgY2h1bmtzLCBjaHVua19lbWJlZGRpbmdzLCBxdWVzdGlvbnMKICAgICAgICApCiAgICAgICAgcmVzdWx0c19zdW1tYXJ5W25hbWVdID0gYWNjdXJhY3kKICAgICAgICBwcmludChmIlxuICAgQWNjdXJhY3k6IHthY2N1cmFjeTouMWZ9JSIpCiAgICAKICAgICMgU3VtbWFyeQogICAgcHJpbnQoIlxuIiArICI9IiAqIDYwKQogICAgcHJpbnQoIlNVTU1BUlkiKQogICAgcHJpbnQoIj0iICogNjApCiAgICBmb3IgbmFtZSwgYWNjdXJhY3kgaW4gcmVzdWx0c19zdW1tYXJ5Lml0ZW1zKCk6CiAgICAgICAgYmFyID0gIuKWiCIgKiBpbnQoYWNjdXJhY3kgLyAxMCkgKyAi4paRIiAqICgxMCAtIGludChhY2N1cmFjeSAvIDEwKSkKICAgICAgICBwcmludChmIiAge25hbWU6MTJzfToge2Jhcn0ge2FjY3VyYWN5Oi4xZn0lIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='
}

for path, content in files.items():
    write_file(path, content)

print('\n✅ All files created!')


## 3. Run Baseline Comparison

This cell runs comparison on the Climate essay with 50 epochs.

In [ ]:
# Run baseline comparison
!python projects/hybrid_titans/compare_baselines.py --essay climate --epochs 50

## 4. Custom Comparison (Optional)

Run the comparison with your own parameters.

In [ ]:
# Custom parameters
# !python projects/hybrid_titans/compare_baselines.py --essay climate --epochs 100 --topk 5